# ROSE-RAG: result verification

**Execution:** Python 3 kernel; cells run in order. No project path,
input upload, API key, network access, or LLM call is needed. All input data are
embedded from the supplied saved results; no synthetic data are substituted.

This notebook checks saved-output integrity, recomputes interval coverage and
certificate exceedances for 6,000 repetitions, and reaggregates selected tables.
It does **not** rerun policy optimization, recompute the true policy risks, recreate
calibration intervals from raw calibration draws, or independently establish model
coverage beyond the recorded interval constraints. It does not provide a new
confidence interval or an independent experimental replication.

Historical RGB scoring and frozen-response cost sensitivity remain separate.
The prior-calibration reference probabilities are not assumed to be ground truth.
Original algorithm notebooks are extracted as a source archive with documentation edits and explicit version 0.1.2 algorithm corrections and the 0.1.3 cell-ID metadata repair;
their presence does not establish that every final CSV was generated by those versions.
Issue 5's exact final-v3 sweep source and the full RGB annotation checkpoint are
not verified here. Issues 10 and 11 are outside this selected reproduction scope.

**Output:** a new timestamped directory `rose_rag_issue12_reproduction_*`, containing
inputs, source archive, CSV tables, figures, provenance, checks, and one `SEND_ME.zip`.
Original files are never overwritten. Required packages: numpy, pandas, matplotlib.
No package is installed automatically. Python 3.11 is recommended.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, gzip, hashlib, json, sys, platform, csv, io, zipfile, traceback
import importlib.metadata
try:
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError("Install numpy, pandas and matplotlib in your Jupyter kernel environment, then restart the kernel.") from exc

RUN_DIR = Path.cwd() / ("rose_rag_issue12_reproduction_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
TABLES = RUN_DIR / "tables"; TABLES.mkdir()
CHECKS = []
def check(name, passed, details=""):
    CHECKS.append({"check": name, "passed": bool(passed), "details": str(details)})
def save_checks():
    pd.DataFrame(CHECKS).to_csv(RUN_DIR / "validation_checks.csv", index=False)
def require(name, passed, details=""):
    check(name, passed, details)
    if not passed:
        save_checks()
        raise ValueError(name + ": " + str(details))
def bools(s):
    x=s.astype(str).str.strip().str.lower()
    require("boolean encoding: "+str(s.name), x.isin(["true","false","0","1"]).all())
    return x.map({"true":True,"false":False,"0":False,"1":True})
def read(name):
    return pd.read_csv(RUN_DIR / name, dtype={"example_key":str,"base_id":str,"cluster_id":str})
def table(frame,name):
    frame.to_csv(TABLES / (name+".csv"),index=False)
    print(name, frame.shape)
    display(frame.head(12))
print("Output:", RUN_DIR)


In [ ]:
ASSETS = {'data/finite_sample_rows.csv': {'data': '...', 'sha256': 'b3fbc4a1317427659e471e3cfa88db221fc64ea38a1b95290837b7c5d02e1b1c', 'bytes': 4515747, 'original': 'results_finite_sample_all_replications.csv', 'original_sha256': 'b3fbc4a1317427659e471e3cfa88db221fc64ea38a1b95290837b7c5d02e1b1c', 'kind': 'unchanged_saved_output'}, 'config/finite_sample_metadata.json': {'data': 'H4sIAAAAAAACA2WPwQ6CMBBE73xFwxnNghaDv2JMU9slaSwt2YJGjf9uKSgHTzuZl3ZmXhljeUDU+ZFVUNVwKHfF5NHoROc1Rj9vjZM2T7YThL01Sg7GuxBhCQCJKGnNhZIvgnniBE8RMFbxIl0O8y1hEdVXcFgRRHFesoIfSaWf5lJBoZNkvEhR9EhBke7LuYJ3rdHoFAqLN7SRwLbh8x4TrmLwFklGPhXHDTTLs663OKAW5O9TWP3btJL/1dn7A7e2dhA9AQAA', 'sha256': '7bfe88b066cc757349ab4bc6b907bad1a001f52d04a5335300cfecf1fa670f08', 'bytes': 317, 'original': 'experiment_metadata.json', 'original_sha256': '7bfe88b066cc757349ab4bc6b907bad1a001f52d04a5335300cfecf1fa670f08', 'kind': 'unchanged_saved_output'}, 'data/misspecification_rows.csv': {'data': '...', 'sha256': '5c852aab88ede904536fcb2c7d6c71f9bae5aaa4f54ef7c2a34ec062edf5e466', 'bytes': 46260148, 'original': 'results_misspecification_all(1).csv', 'original_sha256': '5c852aab88ede904536fcb2c7d6c71f9bae5aaa4f54ef7c2a34ec062edf5e466', 'kind': 'unchanged_saved_output'}, 'data/scalability_rows.csv': {'data': 'H4sIAAAAAAACA+Vcy3JjOXLdz1fUB3Bu4JVAYj2z8Mp2jL1nsCROl9ySKJNU97S/3icfIK8kSsVqkR3jsDqqRSUv7wMHmedkIsHdr+v10+Jxuds8b2/Wu8Xq/unb6ut6v9zd/c8a9ofN7fp+t9it17eLh/X+2+Z2sduv9s8wbe5/WW+X/tf6l9X982p/t3k8WLbbzXYcZX/MDjLDw3q1e96uH9aPuODN5mm92K6f1qv98mbz/Lhf/Lq6v1/u7x7Wy936ZvN4u5tZHu4eT1lX/zhY95v96n759hyv7fMzvXlvdj7c2M/L7W63fPh6fI3R2a/E4g86P+i16XDs028YyMclrrK5sbePR8sQrTEwDxsdSIHkxkbsMHi7hdzWt7vdfrP9DWd92n/DuN3sl3b3X9f3d+u/78wkR5oBz7E329N2819rO+nNant797i6v9v/Jocqtnc3y9vN8m6/3vrV3Pi0ub+7+Q3Q3D8/HK0/rZ7Gy/un5eoWz7Bb4ba2m193J9/4ZbW9W329X8/ffdrePeDOX33GrW8+8HXrQ2MP/NYsT/I47ne/XeOjm+f9zQaA2sQSmw+cDNB/P6+3d2sdMTzvfvtsgzOAn89tN2HiPi63d7ufF79utru9uYkZ9K1xRv3jZrPbL77dYTy3N99+W25+no3d7C9Ffil3AGTXj7h1dZI/mWvCPwDT40+LvEjyL6QaQyiLf908CIB//fcFnu/pfr2Ho556hZ87TDGMz+3y5tvd/a1MA3j8DicLUwgpMwVOvfdMubTY3NpKK11+Silmy7kxq7ElqrXZ5ysVvCOf741yTWZMKdVEIYTYI940Yy811gajnD7URQxTzYljokWZWk+9cqNFn3C6kMUaJ1zGrDhDrB3/hV5ypkrc6yK1Kdj/Mv7NfyL+ZnnT/8n1QyS26+eMMcRp1RyrvNR7xYsY31ypn7D4OWPM1BKXjofNlWtc/OcWXqz/+w5+//KXv30SuZpCZB/5lktUI3WMY5SHjEwxVh/5mEOJilyuxYyce4+KUYax2zk5tFKIg45GoKLGXnPLCjyuhlsHbgGIRceOp94qYQQwKiUbVmnRusDSP8Am+D98KgW5ZKmcMJ96Wv85kGETCrfQ5R4jYZzlflJ4+fOezU5+Nh5/Q2RcPf70fL/afhKXnlJJ6juEieZDGDB5dFzhECElP7CyjWuPvUayGVWBlH489+IAxphaNCdlIJ7Ggc3mba3wuwrH6YWyeVPoAgRP+Ji+SlPs9t4JcCLeOelIr5GKE4eS9CEwcZhqOyAVMJdaDzLFegSQdD2k/kPD5l//7XMwIQTAKbjI02SKZsMTVRtnjLSFPbhGbwX+D0tmxAo1coqIJaRBL2aztYYRMRvCWjdjx+nsKoiZsQAJGQOLbmVCwDXPiX3i2qJEupinTFQOUa+//KkG3gCq4pf8+3OaWiOihqvgjAwQ1n9GMK/qqoar4Eukr7p/jP0tnQaTzUV4YS+JWwS4FioC4kvVyFlyoNb6J8DlKSecC1Mfj4hn1tt8jXi5DtcVBGvzLVyYo9sSxVrNtRDDzVhzaIF1MEqxDzccoeEyYKYzmwu3VsBQ+twZPGm+gPHjEo1okkyjPjXwoQKdJ26MaKBxM8da9KU61vs8x+KICACO5etgGusbpsPnjKkxH52+MWGdGeCoCDH15KVO0uxpsssfuGy5MNmVdIgvuLjFHAxxTBY/MYp4ZDVmkMaItJRVeRQiBlk2wVg+pdMSNBclVim3QBbYgUDYME4ZE0OYjhBlLabW4uG1TzW4g8JvWkmRB+vVrN4U60dQHcNpJ2NlRInYaE575PKJGX+1CwbTch3aE20FP1B2gFBUFYVhV8QEIMwbm1yxcQKYKlqCuhtshYuNBNClpp/tERaLOaSiFAAHyuaVvUCJLiLcZkhFmuBzwnQiKtXTIFDMuU4CkxF5SAJgPEV6r2EynQQPxLOVmTwBcTR9ltoJT5Ouh9NFSC9B34O3h2ZUAZyg70keSyY9cLDDGqhKbxWsRfpYhDtJFtUEa/soiLFUDagV4Ptx4EDTc3rjKU0OTJ0Ar7oKbMZ6GOZ3iY4ds+FHQlndh63PWE3VKA71X/7uAejijiY3lCKcP9fMB2aTKWKXU91b+BMI0kRQ3JEQlDBfGudTzEaXZjZlsZggK6uyGNUueciPGSmB8XQKQLEUky5nGmX4mVRkwuMaNc8JRIMESwrCiKFyBoSCXjG5SsaHKjKpIpCVIM5IJyJmTm/IDRyuwRypp2QH7osYUIsrLL7Y2omLpZM2P3HCY2WckBB4ZHDqB85Jl6M3AQAMgnvxqQXxUH/IBiryRACpQSpn26A/kTZanoBc0TwyT82NbXrJa9Hi54c4DacApB7jERZAupXnMbP5E3RQRs3neVw8I2TS5akt6qQuTM7HBVn1+SYh9dzIH/YsC1VEEfcnCof8jWga/lSQ1Q2MHJuoMh5zw7GJompIVWHJp/jtBVhpQh6ks4pbJuQ4x6wucq2W+wBOSD6+Glif5rco+UtE+uYYBCk7nW1KMVRoi6ZEgMc804TgF4cUhGibXOSLXVI/fR3yROklXupLw4tGBhf89zxbQ+4Rm/Fa9LcVukFsabIBr9BSoHbQ25HZKhJUK0MUKKeWfj90Z6Vs9SrEBo5BAqUqPTaM/4/YRNc0UJ3kr1TKmSbRj/FYlRyBkaEv2axpKuPlQsmPod7BiJ0k5i1aUv0BNeGIvY6UtcwQNEaT4qfWTirugawSJGkaWwm2Wm564mrppA0enaSaipAPsmwqzETcSS0Dh4DpcviI3uol6Y0m5AOIKibBgmr1sywAJQfzUkK2eo4lZggPT6nhKJw8NxO7J28RniTp7bFQSb1pkMRw148gm5GbCSiMdSgM0GbxkiTJ1lie4X10Utf+voSgXoPdIN8RwUTJeQ2ffsQkkitbuTi1c02VIpSFARAhxKbszoQ/OqjIo2ktwC8chaMAlaqOf6vHaNiRvIOkJF6mforqXkGXbLIg8GuJZg5d8zI0oihZ6ftK0F2A67KQClIZMnbuZxoygsxYmOGzLCL4htoQgeGkJtbiXiTmJCm8uxTwgEZHhCE4H8zx4GLDtzRli4f8OsyzOa05i0ZUDvQPWo5XHHQrdyWtBHAWAkakywfmK6325sSHsBlP3FJ5a5O43puuAUH/Y9CyRs0M/Uo4AlkTEovSfpQW2zVoERoa/mTBEk7R6o/YSgRqlshLqZLPtsU0jdW4OrlEFWAOpTCqB0psiXDpgoOgSjD9I6s2VWGkyuZNYSyxi1carChJSpRyOritG2YCbk/mpoCcLPd7eTmtlr21gRMBY0GKzmD03LX+CVgyEiMENEaWQrJssVh8B8LPsWGjiSDq8bi+knqupVcgUkwPUD/PUhvSEiXAViPI0mWN2MNIMqCFJ2KvommcFZFbk7gdGJPVId9H7BhZfSkLKTQzKPBYzYQWqTINRchEWWs8uarxUWT9LiZzMpT+jc3z/vD7cbNfbp8fFzer3frL+h836/Xt+vZLZJz4y1jW/+i997E88+c7935gg3/6G+drxLICMcrRqr9kld4zTQWslGxZG5NZF1rPs0U6FqpAXaPxICKCjUpVm2bSIzQJHZSSLEcTQWAjzRMSwlli8VW01+6RNP9+EdCyNE7IzG7cZPHWF+c5RItzlYVkTl7wlE1utyD21tQbp2ZdFgkkBScrCHB4ppbS93GUgKYNNMuvz7c/rffLMZ/esY55Kb1KX/SQL3bIl2ja6st2vbr5hvn4df33zXb95QZ/73HhL6svj+tf7SOf/PS15/b/qYDC7wSU/3+gHrsYr900BUWSs/psg9uaNhHfzGptRKa2JOWBxDAV1SE17UjC50cZN7MpFlFovjaDVHj081RqZORJDbwta8lDbg0xxsd0ampegbpIuxRuTW8HDwASHwsmEPTWNcSc8bgXa5f6HnYXaJhSxci64psTsRulhU2XLSlBApqxxlhcmUDfW/9aj9lXKat0R3hfD7JR7wrA6zIagKhr1gWdSZgc3ZfAyljukoK7y2bo0tH58YN9U1KH0AJfodyQocx6caSQIhVd6etCfvyZZcnvwXLBvingY5VwsGDMPujCyFmepcg6gnpKFE1p2pikwmPrWpD2vmQZix8nbYZdqRVcTtawGFMHD2sNtwO6LBJgNF/kQ2WIJySH3tOGPJPf62v7gdapPp5NksZMsxUR3KErfWqlcLsmXBdqniqtkfkHQeq7LVZfsYePBPamqJLYmqIYSbIFEanaNV91hUdaAbeGGu3TCegTWVW36XKlLqkgqZc2qWOXFI/FEVjzWL9KIzP9g1unbI7iJgvgrVxmvVN4IqsrJWR1qV28d+oE5PkapCfx0UrshRNT96bg7vRG0KZug9dZp1FuXLxTqkpDnTooR3jUYLfsDVmlDsJsORerx0dZkWVx0JGs5ml02zDQjaN5qg41L6AnyOSMS5QqDSm1e72d34moB9fNh7oDk01ExFDQZvflFERZK37KHZS3V2qnTIjuRYoLmMrgy2aZC/LrLvyA6QYVAH742H/zRVkQ12yeDTW4mgkSBMcYPUWqkpr4kd1bqSAqurVNYfCBlFXX2H03MmaGz3GIFXNyHMjSQ6cTX5IWqRXF7O5bjmlXPy6hpYlGK7HG2pS0sEsfQTdbxXQ3z70EaeSaBdhavEkP/F36e/WHM+Npvgr9JbnPpHcZkdHZYAsl+p13ywIBXXFpwuIo7K0LGExvbAzJGhaTdBcb5ySgZ9oEfFdGY3/i2LTFLY0Cb5EugdHA0XMm7x0u8/WSqHho/qtuE0p6lwFfIRRtFQBJa0sQrzOEpBQ72JusC/NiEF2G8pp0G/o9RlsJhy1oe688E+KYRgrx8W42vFnVFxC0AIp1zYAjRzovg6Ar9dLbrjbpL7OBQDoPxFKcxgoWTfBMC3iwjhXKWI69iCdJxFzoNeWdyXgRbiRHp8Pn2st+YVt4QxDJUmuYtVXBz0dPQy/ai3MunKdv7U+20WYuauIV0joMvbc7ZFCAN84mXZ61peUibcJj3wyCY9c6Nd6wExCkpy8VgUN8gwnSHTZQU8Y8MD4hqUvb8jzoVDpau22eQF6n7nfYAtMOpdnPZ3SYStaWkzteZs/oILcVrIL8lDJfLKP7ELMLpHNIvYyzIEeYLZJARgb2qQcid+1B3gkBEkPq7evrkPXZxKVubTKwJBfztUrZDNM880s+a4sOez8sieVp9HK36hss0pSOHcFvIft+fjcgo6kdUlBZaZ7ld2A4ze/gWphp9McgdrlML2EoRlEDWqgOawomG0V5dR7lk+QkJtRgA4B8nd3NMNJemUByD+RkyKC48ii0UBhZIWVF7igl62hH5UOQPTbnvxpAqkg5F1lJL4wYqCVg//cuim1i6xZoSEIxO/NMlgBa2w+XGygg/zEwXoYOI/K+aluz4F7ReA7CpHl3aOmjrgVEezKPzPjDdjlV7uQFFiFTUzuQ8W6EMinBrtPhxtYcw1DWDPmBdH2sPiKrduTaVMbLcNgH9b4DDucb8AX/nea/tYNHD38NapzIO+ZEWIKr6ZjvUchW/InSr66ryBdB9WxivEa9E8xoEMpiSvEqGVIdyEklxnJ0WGCdNRlG9u69uQRVZKVEriX7Kj0wttkvdczi4ZsgeVknUC4ldC13epqXDyUZHquasj/UbZehR28s7lBs3TfpRbC/8TwiDdd0RSe9bLETQ05Fm8qSLHd7XdPjZkei2j0OVebumwuDBkmlvCz9YppSyRq51zWDSxa8O6gxe1YPDkX0lmKaKVMwIx+rnZ72peOOtx/eJFptBpFQNaYdv6BDS5wku6v5U1sPP0TlkrXO7H0ZWZI3diNCnY4vt8bDJttlknmeNZRJXTMnK1GIIA3DGLvn26pO3IhAMTiGVLkoNB4gX7Tpx9le699d5xT1o5oKd9MzaH7WTKwbia0EhOgZrwbThWqc8J7m8QDPVKzGibntxSlkf3YcS1rmuyOrNdtL3VP0qZaooR69xIlc0bcaEmKjtZ9KjuglMMQf4el5cueNV+JVng/8M+4OTcn2wsreZCvcX7bC+RZtugLHSR+hogNVzX0Yc7XcAb9BWCP3a93KMolz8cxB2N6jY6mpuf6U+q9FYUg83+MdkMl3E/BVuhezsJx7ZPbday/qmxdd1Ysu27Lcpk1rydejrYlACSP57VdkObosyyGntmILnsdzazxg8moYj7VZLT9bxyCHNPgMLqfHAe/iWQj0qCeAgRqCmK9ClfFdCbJBXFnOc4Z3eK7E94Ppd3jOc9Uo++sLp/kWe9/V27VZ9cIBlK7AczlIpKi++s02/WHEKJm2CD7sGaF1VE6knOS9l/AazzGaLcLCVDyhE7q0w8AlbpNvHtAtafGwW2ZKIzHnsUU0TTR2Z5xAp1p3MT7xeqfTW6y8hANRC0RmGTqojk3GyrzkzleD6jJcV2TNw5Y3GLmZ769O0OBW8YRvV1vQk9BfbbErgfei76GADPQCC0KV2+Aydsoci65B605eX0VBKgw0hO0OG2XKsYsYhHdsL85Tysf9Fuklw/EMk7HhgnwPaLPexe5vvWIzX5qQRUTMvBmbsS9eFXhfpM+s152dsvE16Kx3EyDgmj4KlFIusRIgNAh4yHeaIO2yoAN5FlsahNZBFlofw4H+rTE4KhqhcYzdeyhk77NV0pIwQ1Q+6yM+1lneNuO0cNHCZrY4kVi+zYjzoQrtzRnUYr0ip/FlOU2/qUG1lgsOqfwfKpBhVDUZWNp2eNmc4d8pg+ewXfi4Ixo2yQaDZQayFUCNhbXMJkE4STvhK04bG0PL4bt98msfPJPQ/NtTMJyI/C/yAeQHGknkW4u07px+f0zkK3ynQWBpuPSMzMqsGNvkawRs3xoTZcsAYNHpB0HRzYZB9x0K8Hv7KCa8o0hkgQXTTL6HwpxMvq9CSloDBTqUPuYrp/n4PRQnkMAI1Kid9ul1X+pbYMYA45nwmGG+YNqafyOALOZrJe4y0Fyq/aRICVmL8vIlPqYCinzHlUmN0ZIiC/k2yZFZJ5cfIg9NvcvSh7ekUJaNEt2Gw5ftZLtctYStRJLCFejqMPrgK5cTYfY9Pnl62831OhsbOz0HMX0/83qXq9rYOhxAMu0HYHqHmv4X8H/6ZhdSAAA=', 'sha256': '0fc596cffe967aab84f4940fb70c1e2383b827ea82bc1ec47e1f6f5969a7cff5', 'bytes': 21015, 'original': 'solver_scalability_results(1).csv', 'original_sha256': '0fc596cffe967aab84f4940fb70c1e2383b827ea82bc1ec47e1f6f5969a7cff5', 'kind': 'unchanged_saved_output'}, 'reference/issue5_summary.csv': {'data': 'H4sIAAAAAAACA91d224kuZF9n2+pLTCC98eFsYYXC6yBGb83NGp5RnBbWksaz/rvN8jMSl6SRaa2S6rIKmIMN1WXPLycIBmHEfd33x5/frl7e3x++vL14fXxl6fD3c+vbw9Pseb++fXtEP7ny+v93beHw98f3n59/np4+vKP3x5ewzteD7+/PD/98uXu6fX3h5cv9EUP+efjv++f//nwcvcL/eH+/reXu/t/He6mivAtL48Pr8u/Xx7e6N//fPj65e35bw9P2R/ufv9yd/+P3x5fH9Nznf4YH+3r+b+/Pb/dffvy8vj6tx9+vvt293RPb75PsA/iCPQf6sN/P//98enu208P9Fz0/HffDspr+pOQrizh7UYVRVKdlbooVGV8XuwBjsJiXtwBDL1NZCX8IggtfF5CpUCD5adDrbHK6lTcAOSf/vDjj3/+6T9O0NCVJVQqK4pCVRp9XgI0TQ+ZlfDt1eug1BGK59UTDCmsJICnEhELoa1xhPBUws8Kq70U0sBccADux4f7t9tF99MbVdyP8ZWDJOJbDRvlXVbC8JIKinJQ7iglZCUCoSpAt7ymgSkcWpe1TXyrNTRXpD29/LDvpsn/79+WeaexLOFrJeiihEnmVFFClcSiHORRHDSYoxOYSpzb9H+o6dWpqAjIai1UmpLhV4QH7YVbhsQAzh+en/76+PWB3vCXX18eXn99/vb1BMuZosShUNFJeJ8v6SQ8WAU0dvfBOEKRgZ0QKEIlnT2VqZ8g1GeE5CIuRb2kaCBOZQTsj4//+/D1L8//819LL7mixLFUdVx4pMBTWYm9BGU5IPWSVIY62acSB7oArd3yjFPX0eSSzmfcFx/HG9BCLM07Gnb/+fT28EKs/+enb/+aEUHF42Z6grIESN4WxcUeKl/ho9V/IG1Zek/Ys0qqLKHdM94JxUYTpLISPmkKcxMJyqUJEMsBwB6t0qlM5OZCy6YR6OehRs+SfTq+ld6mwSc71wfJ1ioBUP9nQ3lmNmpaWKZWHxpfm/T92PhaJDBam2wexoezFumtfUgseRtp5oGXS5mITlMD6a0zjJN5pe8NgPBU4ju9pz7GxRYMWJGPFSIotLgpm4xmJzgv0mqvD4a1DaItyzttULFxiSPIrzYzRhcl2iCo+C3YICwoYFqbOWfTstLas5WWGML7pbdUHyNbE9SstZasuFNiLtDHxtcGXQDcViO0U3gsDVKr0lupvE2k3ofFh8BblUDrBEMr5xPz4Gh+8TGurUqA8H+zedGHw9oeYdceVSUapHwU0JYktrrIy2SQyr1TqCrhHbw7umorRdshmuQ+MUY0UUDrNZ31VuwsqxCpbumXPsLKGlVDIq6XywkTOkSXC2YdCW11jFjNFkO/J/MTRDufWIG2qxFHq23QRqhUIjTaCQZ482sArTZGt4RtbYtuCR1LU+TojTY7/56W39IYWBrP9WHxMUWKOgmy45nI3ZLImxDCXPpY2FM3J1MZpgONrfw0CwLK7FSkMyNU1xA1pn1VF9ebxuYlrkrXU6Q61dPUW/JoqmabJrlSNDGyde580kMbVJUviadlqKRF8cmnNQC6wR5tPOWpTyPr8SSs11kx/efaYkyu8mDbLMFVHo0jjQPRndZpvptI4156l46bdR8Wa+pTrOyME4b4Z9mqxSMrMi1SZRw1mndsaJxW24RHZV522rUL+mrjR746HDj+K0fJ1E66LC3viThaI/MSPf+V14X65Wghe8XJ4T1YAe5UprUbKusxc8bEwegdSpN6doBxN/7IcDRMvZys4gBYZZkaXYbFZs+FKuvLTWFUb+SeBVDNvpY0Mou+nmDQA+tM5jEvuJWmZkxvlREcGosutesAXG3ebgvd2kbeWu/xljQAShC0XBgKUHAXrn+grrFajk8g8Xquf58X1XvAzzFLjckEaKujhtnDQkvL1IvTfkN69DYtgCY9iScbItOCtQ9yN/5WCERhIIHt4+JrlmhVhFnjTCttSVygl12j62NjbJW+Hxxro3SJvuOtAwBJG0HiptOJSB8NcxmAlBYULuQy6BrO9mhPnkiaEQ6UTMdqfVhsmbpVCwQ47AYXb1YfG1+mvgA4zkx9AXjnl39YlLgzKc7/dDwDcbmzJ5KErQV1wfFUloPDSq4zLbel1knMaeW5ShAW6FfSp/sQuTMec5GCktYjbAXDXqSgwjSDpUPPo8GPZu2yHCQciyP42ALBq0KPu1KjarL42eyMk0JbabIG7CP7WM6+KjTOjC2N0Bk7TEOSJjwR25YDEnwvY7u1CsCV9Bw37OVkDN8mVx5C6Y8qHRmfDo2lFLSYTmq2qRONEZgLW6ZeNCooVpbP92HuRwZgiTGtPr1MHxZnY7QDmYIhAJm+SY9IjrcMwNIqymUrqPNoujIAU75si4LDQV1R4l3SWurzvfRmEUS2E532SQ5RyA0GV23m7oaT25YahwCl0uJOe+WyyAPWx5Xzyhq9NznJTGtR8C4f6FH0iGi83MJqatuColK76wiuELLHA9WVkBAqn74J2EyhVJyh0Tdki3E1H3Ub9Bm7T9C0tHZRLveRbVlQ7BTabqQMQBaevv80ugdjkbMpYq9koGZA7ZOZMKPZwVvJEJkvPdLZCSF35+cntjA2fXwA7LyNXUOAaomMEcPqbdblq+Y4ynxpnd2BJu7R5zNjUmkYRwug5dvsrNKgoZcN5ckISaJIlRp9gLK2Qg1s5UPHxX6jylZN8DEOcfBUpcf7I3lGznBb8BpLpNsCeAW7VB1ZDgcYc8WFVgYB0jJ3gIe74sLEqy6LT9z04OzK1+/Q0n+nPoY+rndt/3T5Mu15YMtXnC3lpSUyTUoefTEeppMQaxVWt5QUwYZSvAFAq2yhlq/rQ2Rsl1o+cepUL3G8fZdtPcNNoeNtlb4fH2+jtAPFBVkkS38YeSXlHhQXhraDfsM+W+5L0RDO7mimLPcD+7DO26MS1XThaXW7xK2vwjZ2SqsL9UYfXX5cN0MMNi67ZX+uEogfPIi0pepD5GuPmp5/FN4rlWLU9MExtkcXQMfaHl0AH2d7tANRgzUSrFpGgx5NFeaiBhtOxeWG/d6uPKtAOwytljaBPiy+ZN10+iPZWpvNij44xmR9AXSsyfoC+N61VKocmdF1W1KGat+xwFJj481BGuL6/OrodBYR5lq+xY8HL8GfVLmCUWlPjX86Wupj5GyQdiBsoIa2xIDD+4hyD8IGFNRUtEMaamLkvvyrSB9xkA7M+rA+Z9ZfXrCBBoM2bzkV7KPka3bb2Gg9ASMtimwrGm4HGmuD+73gOFsi/roGaUK4pnTZcjRHeOsaCEx0QMzl7D5PDdz/aThO5YzfxcvyFc+5yssPjSDEFo6u7lmySvTJzNF/0qWBrONrQXgSB+O7sKrp/fd1+ogzEe4a73PryDIfcR+eNoGQTPEAXc3aNwZvzdw3BnA/0qIQmIdm41I5AHYNs1QKLtVw6jDXNThD7Z2S4pgBHu66BrKyqN04frfq+/8va5tWMnGaJqVsc5omxslq1RcCESuoVdgQ/Gcmi23cR8nZODXvxXu/6XacamoAbgwec+P0/QD3oy0CZenbEwP0cTG3TTuQN3hJvW4WP3UfDXN5g3fWKLskouuMnYaTnNOMb/pZQTgv1Thchmp6yW8MHm/GvgDAs6umJs51ohG/vp/UYHpZzEtCqeTRFtP3dFlW5ckf1Nla4u+QOWm5I4B9lPuRGBGFSTN0KamraRw2W6U96AJQhIZPS3rZh8Nbs0FgnEJzOjUTnSmBvM1S06MMBM2IZUTYPjrOZukC8HibpQsAPG+W3CpDXTPT0OoKWRPnxSUBITup1+PgQmpfSqMQCVikzvN9WKztEn+tAyKNa6eHl6/VLqQOMkxTWBTKnaGjPoXY1qF33Crxd4i8Y0Q11WgnGC6XlsFiqPMgqB+GUe5USwxwWTKrE9NQ74UIPatrszTeYJ2ZxtCWFsuAPwTOa4FLyvVB361s7k2h+ySLdHlpAH0jyCUfJPRR7kdqRH9ENw50o64medhskfhrHoyXIfDyaYSPaIC35CHEEUC5aO/OdoxuqwEYzfb2ZXKpHfjN8D6YtK+Or7GguDGE5wxTlcg35lEVVTD6GN3OVxAi0PLALszXSoFp6Geq5OBzfEIpdZZPZQbqnJTVVep4hqdSnFMcAN2P8oGYiPa64yza+nrKh8qzPWQJ3soHBGs0JDXYCA9z5QOiQQ2LcMH10PC2UC2XMoRNol6GrO+jY22gvh8ec/v0/QA/yzzVIVqVbm3spSQ6Kek+bBkdYNUihNLFIHijrHJ6Z8IHDIEdUuwb1cfF3DTxFz4gPQBNrqG6S+9A+IAS0Ifj/bl07BJ8+GHXBzjOlZfSpafpo+Nsly4Aj7ddugDAz7JLtapUNWFKcDJvEnu2FojoYUscBL0v5QN9QoRA4SOFmL6a9GGzXdqF9EGFapNiDffhMJc+KJDG+qUZ5XkwyNsLI6l38/tL04GQofHmYZh8XrelD7cFj7dhaksfTEjPsUwM6AN8n2GqklpMhqnIh30GZ3bEH8tB0dQsXKhzckma6vk2avanBcdn9ulpa0Etkt927ePcj/jB0ubQpOBcfVisLdMOxA8GyDKmSGcjruMtfgih+pNkwJwHs0dxgLfW6cQBfXC7Ewe8C92HG6Ur49ut+AGFVJD0eAOYu1E/SGI2m2I/yj4s1iaJvfpB0nCjnkw+2hHRsZY/RCWQXRIVww9Pd2+/vdx9+/I4D5IvDy8vzy+v4eODmA8tFV59ThKPNcs57VoJL6mqZhIHR7WOZBqmvSqzNBEf0E9BFgY16nMtTXxIeZU2IK0McCOLklpzlq5y78T4FXUw13p+CFskCjXjZ6vtJ6uHW5s/Zm23npJVUe+YkqWy6wLeaxEcwPSPFAV+CKlnqdb2ZqulWocc/m4thPAogTZQpxnrx9iYO+cJIWiRgpRtQHSNTFLlIYAdPOQn0nwRjIFovh53M81biTX3E92H9W2RYJP6hAaiT/LWEU62JM+Z43lTPFM6bLnfhQ+ry4y9NgwKPsar5eAWtLWjjVcWFX+EiPntfgFAfJJFnRnh4c7u3WveTXav6D3ydknv5szUrsaXCfkIfLH1n+ZLSESZNZA+X2udA8CNa6OG7IALTbU0A5yejS/Bj9zYVyP4VqXXEn2WxmUEjHkegHBt1/iUDH3DEOdjrppnImS1EFJQsfHQ407v3evSrbNYVTV05PJ1f6h1FGaxWtBj7YKcrzXQztbW04W2gtRw5Q6B7K8ha5v6eAiVKb0jY3pH1vSOXOm94QsW3kul/XLQOEbGPKy+JE4nkIv0YgRnD2TIyQQ1nMLhsBppFJ0epk95ii/ldR2DnB6SKy8r1rysmPJy0yEqiBholZICOY2QcScy9m7RYC6kUqnHNkxDNrzcvBgeFsvGLB7IXv/g7rwwLkiq0pgYYzvL7I1BiMKI+q6V9bU8xVbzK1RB+ToA2KMvRu/kNANRR16JKVhWQVqARq9VYpmHZgyUu3cwnK46Wn9uIG1s+rGrvoH3sEaZuCPSb5XLA4lKqkPEabcHSuT3xR01hc7JBh/+TUyyUW18Uv5s6rEq8cytAWykn7k1iNxjz6MzVieLh2NE11hRlBw7eMZducm8pTGQvAZDaOeTia1nDtbXuYIlWnuvbcn4cYldHnbJg5NB5Fspj8j8CC+rpbhA6up89Npp3Wq0x2UwDGEypXZLve1SUYkWLObK+RE8rsx+KXx8if1yPcjbXR2IXZKVOEWsNkNEzN3VUlrq5yzc9wgPczu1J28f7TasV2kw6SGy82aqtiwhqE1Z4oZpncumjKMet1Wo6k6R9qiKH5i2UKiEzoSzcK6S/qm0zm58DYEyNVTnagPPeZuuaY/HKFdTdTmEfI3V5TDugAzZe9SVtNL4LOvvCBFzzQPhKVMe9fDsyXsJ1ND0X/qZIbLz5mqNoZ4iMfftaobYdZY1qCM3S6jDp87nfSH+TPbW6X4y2TWdT6kI1GmQYolXOoL50UzX3ISumksfhc8LzGvwwHWiSjY8k53Ol+xDmLVNXi85at+TiRhNzbMtjDXNVf4pM3y6lUHl9XjMLcUOtAaGMDiRhWwejwjejvxwAdeZ5cOyC2hX7lQgOpEyZSAYAXvX6VuVe9O0rYJZ+7yFKwscfB32f6JLa8wqUgZ1q1FZYujpeAMs/SGbm0OkfDlU8ebQttiB2QMyJ3n+keKDm9Yn9t0yYnmLAsL3QKLY3sGH3JkogLYCSD+4ycclB6KABop1zAqCIcsSHS+yzrdk6hTLAL6MajQdAYQfysUCp2iuYArt8pzuVXXPRuVZz2vjYK5EEd1HFdiAooqN0QzhBPboCrBzbyHm83VWVaPWplJkA41Pb8Um4ZNsqwMa+Cr3WOym9f0fs9abeSyKDPjMOn6H8LTgyJNjzPjCsK3y60itPSaS2YBvgw3cN8BrhPMoy5ZOYC6q0Wiolxbh0YZm565YMCGws1440gwQ7UgNADSHtUg3v/0Q2nk7tb7UAtXFdRPnREMgsLJTqxB6Bqpm0qcEtphLeGf5mtGQXXGXU0oMMiTCLS0lh0j5srkKp4/p4idONOUkwKbrE7ItCLgxfNvWG/9/hLUmnBBWspYZokWo0rMEiM4Il17+BNFnFDuEyN5c7UC0YEK1XAbbeFwxFy0Y2jl6fcpBIPs7j30pAhxq5zY5yuVAEdAwVY0cl2uPaWMvspYSSH0U+ZumhQVCGF2ZUTpbS19iqetSMw2B8qa6Ri0QUtQ2S5Uxgsic6mDT+SX60g0VeqByTIUeWM8wkUdrC4V+sMGPIsS1h+JsTp+vBi1CdJGF+sYItywZdg+RuSTABnsCW/wHcg+KAGvI9qplKOgunl0pAmh9YTX45TVE9i5zVSdbJgz10V6g9/JS4sfEYEfpBJmD5WeGQJlz+UZ/fj7mwWNkgMKJ5uOUKcQF2Ho2XR3dDh9uiyvqek+3zRN1vedj7sr3Rlkik8Uht2E0sHblY4hsIrPYU11Au3LlIzWCzXIVDZHtgOGbkb6tImO2QfIlrxYT+x30rjjTu2JN74o5vfN34ktjiRaTs3XDYODtxDfUpFYsH+6pfNTA0d24gCFsWQJD1CfLTW5fbUM1Hn1BLfPNQW8yezUnLRVW5ceEMOlerQPYFH1O7U2sEPbHIZPSqYyxNZJyrLtlcxKHlUMIaqsmaewWr9mZb5Qq5Evz2TqhqdgnVvjs5s0YY2kjmimiVKMjvV6djDU6TVRCPXsAWedCnMAYl7sE8KRatqjq1FE0P7U1aS9txhgrU3ObIK+xHimJVm3pCeZefUezymS2fgyJu1efzDBqtykaseq7vi+dqqd8ubCA142MRC7PnabmSxvCA8qq+cLJKNAmbPnGIdT9CBhodxIUJ8uPD6Hxtl0OsRbAke0ykMcCHUJkbbqIDvPFFc5LSxqpLt20HULkbbkug5G/4dqBf98D9Z9akosNATF373tHqx+78EN/+QN7yS/XTjqOIPQY4G70C6BsCE66DKUhsg+m8bohAI+qrJspzhid28eztaHDrFj60Q4BfjSJXx/htuyV2hQlHnNUTNzm74bqXYWseqsbWoJsgMlXOWdriTZp3eKWQOR6iJG7mdqDWx9FWALicj6KQ0TM08YLR2vj0z0NL/p4cC92qp2DXHpadWzjhH0pGJyV3m08amm6fi/O6LWrAYAaA7IyU7cKN8vqoKbonKxlxUgdh2qTqFA1lAGAdRrRlpKJ8K2rXH5jVgl1LpoBVDc0bYhmoHKtYh7MIITRL84JQMZeTqiHCGurfIMQuZusHSSTR6Jip7fo0NQuZAqSxpnz6Wm6gHaduJu+EWKu4mFaR7UzPYaxlszC6TVGtoHLKwaDaKtWmgtX8eH33UZXLRkAs6dbm3pez8ed4flLFQwtrIhVNsh+1C6UCuGWe/CCzKW3Ttd9pUJzVYvV2hTjQJL11foGw2PVd/Yg/bHo4iXKcfEL84IosGB2ejDfyQ/b+mUAj6HuSKzgyH65TTlR9RmxAojG1mIjVVVp2qJGpVq9hk0JFqN3kZU4XzVRWNDSkMhdPnNs4BB8ICVpHqP8VFv2DqrTbaUBt+e7hr2oxtKWRuQtEiB6NZLMx2ly4hgSc5EAEtGGiL0zMY2Y9PNsRgg7jBXvTD7yIB9P5xn6FPVEmlpJE5bNWiQjM8S2I1GAB+tTCIohso82EetZ6o++3NfNxhylzs+z5uAmtCgoXASTidCgks5xDJKzhWBuIPZgH/j74hElrd7MlpQcege+eAwxc0U6Cu0bh54v/sLGAapJQRsKc1w1XKROs86w0qoFajG17eq93pdTHmlD4WDTsZ9ue3TbpmJ9197X53gBUDF0I4GoVVv4tvJIuSAXS0Wfrw4nYwrTjJZDlHxtBXz60di7H4+7rdiFQ1yGc/JtOTD1HhziCqSxyc3Zn4Kf6hCvBpyUR3r89JqdpzRTsCI5Mn2W6nKrRYbC0TZq01U8vS9vOA0XoVTSygyR8eVQ5M2heAX/wrsfkDvJ78CFbIC2u3a5/7lhyPJ2IcdfSWK+fv/s2oWMQirQqeOHSHfjQ5b0GYtb5OL6813I7yIpxZvkFXeSVzsgefZeZBkWjlqljfCGIcvajSzROWfl8izmh/8D1gPdcGJAAQA=', 'sha256': '53a50f899880bb50415804c6cd16dbd46e5db2c395276ec01bcbba05ab832db1', 'bytes': 82018, 'original': 'issue5_summary_by_method(2).csv', 'original_sha256': '53a50f899880bb50415804c6cd16dbd46e5db2c395276ec01bcbba05ab832db1', 'kind': 'unchanged_saved_output'}, 'config/issue5_metadata.json': {'data': 'H4sIAAAAAAACA6VUS08bMRC+8yusvZYk3jygpCdAtLSVihToqaos7+4kceO1F3s2ECH+O2N7N6RqOVTd03ge3zfPfTpiLFPet5DNWeZgq+ABnIgaMcuOg9mjXAXzJL5ca0Tj7FLpGLJURurOD6Ai1ZiPT/hpfhp1ta1AB78KoCGHzagXBtvpgHBSaCHLDZhKBPfgbRswUonS1o1EVRBVdCtluQbRWK3KXXBLb1yDEQbwwbrNoZ+3rSvBk+MPUpJ69N2D8yPplDSjxc3t1WBx/mnkrAfh5Eq4VSFiOcK22LToR1rXImIN/b1WCBnh/IwMCHUDTmLrQr58yKNWy7qopFg5Ve1Z+TA/7oRxL0x6YdoLsz3yStb1HxDj2d4xCXliZGzcC9Mh32MYUViLHp1sCCTn9EV9BY22O0n9FDXg2lYHzflsEBwVf2P0LkuQ2Uf1CNWdbb72igWgU7CFc6171aU1S1WBKeFu7cCvra560zdbh3bewn0LBpXcx9wiTbUME+g115eLw+cCSozvg4Z7FCj9JqQ8PZt1a0OjmxbCWASqeBN24u9DDoIgQVw90txUTflML8SF1JISr8Sl1KqgcSprxJe22VErrttChF4MVbMzRXbAJ7v9EJVybzO+vVbZb7n/H9YIXuvpTUVfVflaVeKMZz37N8ouprTUf1l4DJOkLvX8ibK0xqtoE9tJd4MhgA58S5dM2RDTU5ptLDzdZkRN98Pfd6NHS3+CREeVibC8Iup8csy7K8pUWNjGAaYCqZLXXkjNPEUoVFuFO+apE8Bk02gFFftlKVTvGFpWWFwzl5Za6kFgZaWF5VKVimD8B0arxSRbKkSKtPHmiY3waR9VLdM/4TlV7IDelZAoWixDQuFXOMj5gJ/c8dP5eDqfnA3P8nwyyd9xPuc8O3p+AS/tvF6BBQAA', 'sha256': '98d2236df287dcfa1abe1e028719c6d2016342408015dcb7843a3107d0b42b7c', 'bytes': 1409, 'original': 'issue5_metadata(2).json', 'original_sha256': '98d2236df287dcfa1abe1e028719c6d2016342408015dcb7843a3107d0b42b7c', 'kind': 'unchanged_saved_output'}, 'data/prior_certificate_rows.csv': {'data': '...', 'sha256': '5d2ce2f35f5bc74ebae9940243269b394a2ea85d117dd714d46b45ed6167a948', 'bytes': 9269344, 'original': 'issue6_stage4c_task_level.csv', 'original_sha256': '5d2ce2f35f5bc74ebae9940243269b394a2ea85d117dd714d46b45ed6167a948', 'kind': 'unchanged_saved_output'}, 'config/prior_certificate_metadata.json': {'data': 'H4sIAAAAAAACA61V328bNwx+719B+Kkd/OPceSm6rg9dMawNMAxJtr0Ug8DT0bYQnXSlJDvesP99lO4uuRpZtocFBiJQ5MePHynen88AZiHijmbfwuxjCIngAlZwk02weS/HrbmjZtF5a/QJNHE0W6Mx0uKA1jQmniBEphAgUoizeQH0iTWpw0Y5H6n2/jajr34NxGGFbNCtrn+++WFx/e7HclByUCX5haor9ZMJWrBrxmi8UzfkgonmIKnUZepOkfhDqgV9abqTq4eUmfCmVuj03vM/p2MfSDHulLlPFyb4PsUuxbDaGodWUqxG3L1mxd5HNVFgNWDc34kiiVTEcKssHcgudTj09Hpc1Zj/nZpWpUGqb9CUnhob1FNwPTHtndiyruyPQdi8rF69riYe2bZ5/c1g0TYFETwbv64uBmNDwexcCR7dRtSgOuLBId+vi4OMQJQpwU4xdRR7T7leV/J35hGImkLr5UX1ar05u03OxCxhjYEWn5OMnEDBwLIvVPu2lfIeYhrGY1CoRWD5Z+2ErUBFTtRXkVpio1X0lljmKD+JNS3WfY0BW1IdG8/KOPqcRFtpyrlvX0yLd2riL42Kgtsal0IeFvGtlg+eu0yRhHXnQ98ZrIMiZj/1NA25mFOKAlvPrdSXXKO2aGxikpKSy8JUgwTIMkzfr6v/5N6icUpKKgmyukxbYpKyhgpGEKEKbyGkTl3B5fPOqA/vr59LUwI1L+Zw9QZyg4fQF7CA36YOs3MdA7pckN6TLgviSdxyhu/ePoZJd6jjlxr+exWfLtVNaUzOl8HenJF/5L7k/F3uPn3B4innnqFxMp4dU8SR3DvoqR4InuLZEjpZrXuMc5BBj3uCrKBsZdGOIFdsSZZx6xuygG1tdinv5LI++TQvEZ1F54in27tEpkgNbNm3g75gAsjGBoSBAqROnjOU2QExFDQJ+IMcSP0wfBWOe3KF42gQHMrLEHOCFIzbTSdjCb/sxUN+CKXiRSsrryPdc8sP+rFvyxI+xgeG2qJp+6SFVNFjIdqIHtnimVo4YoCD8bbwGPknd+v80YnsaOHqqznUpFHKL5djpOTJz1bC7peFeI8dEFmskbu6AhRl+u/d0APtD7ISdrScPfvrb8hb0tVfBwAA', 'sha256': '71beb169c99a31b7178e8c78092f92147ee4f716c4b1255c75f1752d1fd0f8e4', 'bytes': 1887, 'original': 'issue6_stage4c_metadata.json', 'original_sha256': '71beb169c99a31b7178e8c78092f92147ee4f716c4b1255c75f1752d1fd0f8e4', 'kind': 'unchanged_saved_output'}, 'reference/prior_sensitivity_summary.csv': {'data': 'H4sIAAAAAAACA+2dWW/sNpaA3+evjFzgdrg8djdmMIMBpoF0vxsVu5IYcex02bd78u/nHErlEhe5pFsLyXKJyEWs2sRPFM/h2fiwfn76cbt+f3p9uX/cvD39/NI9vL48PvkTT4/d75vt+7ftj/0b1tvful/Xv/++7h43z+9rfOd6+7a5//Hp5a37bfP+y+tj93L/j2+bN3r3W/ev7evLz/frl7d/bbb3+Bubbv3j2/vmxX+X//vh9Z+b7fpnfOHh4dt2/fBHt+5P0LdsnzZvH39vN+/49z83j/fvr79uXvYvrB/+8e3prb/eh9e3948X3l/f18/326e3X/Ha1viDm582283Lw+b+9+3T67Y/+e0Nv3H0t//fe7zM+7dfnn56H598fv356b0//W8/rp/X+E2P9w97fDsa93++42z4o+s6zlas+8vry09Pj/Tjf/9lu3n75fX5sVMOOnzN6qAZOqdV0Oh9TsK4KTxlVdDoXarTdsWk2Dd6I2OKcS2t2TWHJzlnwjjG1dDo41YayRjTIBRjztn8KUb/LUbwv6+/Pb2sn/+2wTuLI2D9vAMQ9l/7a1Nu3PylAQsanjKWjxt9UBkWtE4D9kDv24ADALQTH83jsMxx9gHdnBnHf/3lhx/++rf/+ICQ3nEeds8K+vn0lIFwZNAnDYStk3LFlBg1jwG7LJDDrhn6Oi6wW0rZ3eHOzOGHzcP7DQR+5m/v+JaHGwr8zH+/vG+2OFf89eX5jwEGDzskNX2xiBr1wJmg+etS4bG7ruG/jxlwaGfu3H8+/d/m8e+vv//P7jaDDRrBlQKCRtekjRo3ugDNw9YJ/H6p9Epyt289KQ5gJds1f1eNltyojwEGZ3/Sewn+p+ePqR+ifkrqO4egUT8jSUenxjIOW4eDuQOuV3Y8Iug3JHVe70e58l03wggp+dBO23Ox77j4RPBxJsNG3x0PRuqCSweItWLc6BTI8OgsTSHjwz/fIIRR4UnOBeCD7+TQpp4AoYUQTknT/+s/6RQqEcI61CqcAtndoQwGbkEqaS1DlQNH2EJekWQciWlqnpKxYSNIgo+bv6UBI+sHVzRZys6olfnoed93xqQDwWiIDK2fA4GGzegyCjKKpeYNUgZSKlFvmDKYptcm5YXuhRBUqXBcqO/N6SMXm2JbU1fOBQb2YKBtW4bi2lmyAiCSHjdTEtUrgxqQMExw5fxzDtLDZAbAMKXlQkLTpo7w8LcoHD4g8jBObOvA39buw5iiy+KKTSEZSJk5NpqLJ1a90ZSEDx6+TyZCGoHw+FHk+DRpY7XYtbKUEkPJDVMOU8aMcgOVA1WlznOhvjen81xsjmlN5zktmLc/3t43v9EEcn//K6pKwO7vH/+dvnT0Etla/Kj+TBMaLfh880+XCRtd/3428M1LejduNHScDprpuMGbkEw6DgmMzEnczznSEKKPlsdohJKcGQ7+LPRGIbBcITflLGoc+NjfcbxTOAScAdS2rNFKnYRiLP5DInTjuYOgeXKoB44adSKdYo5ZouP0p8XHt8kKwSUawY3cTHIZJSHHLiPlUiVh/Bz2WIDxceuMXbnoe3CWcqAZ6I/myeGzrcVOCtX4rE6ubFRmmSeikyavU0XLGN5xv8gbD1FO8nQ8jntbtjMW53770fwky7gjNWtopkKINWpeRYG0po4VnvUb09EuQwsytCBVLjKxJixq1ItEBbOpr+4IEcmEUyhe+NCmVoLMaOeYEjTHSW/oYlp6aqTTKQFA9jwIuYmTcEt0ixu4eeAyqsUN3Tx0k8pFOM0bNWFqdlwGzfcwbDQd8fDoHAoHF0sHvGsKYHSy99gbjbI5WNVVxvATZ160Il2kYoTyxU/hoY9UiI67VeY+MS6YHGvIvbqjAo62PpBVmscK8mjOZFZUeLZmRjs7rLscrLsDU1Z2VQ7hofMPV+Iv4ysVx0kK0Cs1uilDAJYSwu5vgRusaF5j/fzx1OR3xS/Gy8G74GcCwZUEK3CucxKfEaVJ0Y0wnobi9Mo8OgiYABU0bwMKzvQL80RHiYOwObcrF2D1cz72GDl8HP6+CYdqiQse5doYxkrv2JlKzU9xYdN+igua74ZKjGwqsuIC2YXCOXQwDAnB9L55wWPxCRwJ0QrZRQutLLp50iFU2fy7mB033lkR2d360EYLBvYCpDcHofzVe8WwxlGXrhgivd9fj4ziBPwgC6WFF61JLDjEyojlKxMI4B6eRh13NLB7pUQawz7WJKZCeDWqaUWBtKanFZ7wG1PUzkrLQN7waCYcdZm4ithRmTVimFA2mGOzU3CqN1YeWkwiHhSnnHzDDrWZXjcCHJnMB00Zx/rwOMCBK5xFSYJyBbmeht2c6J0bvAl482J6bvgm8E2HI2YSzDQErceoxs0vr1LYOlysmY4zjauK8Sd7wxoupNzITAdDghm5jXe/qmvEuDTaI9VCbMYfGlgevTR18XDk2q5UtCRAJBoXpPu0hWGdiutX+2GU4zVirNL7WZRIc+7PwpK0Nf/nRXBBDtdMB6gLm3dGpadEeBwrOZkUXI+cMROWSRRBwuIE61Cj5axfHguncT5A7hTuZjnBYyE8lD2nwTfLD3rjN8lvpjv0RnCS4DKvaJo54nL5JYl1UndOoOYRC2EG+HEV0sSecZQSB+PYC3O7Lk9oYZhVekMLM2nOI1pclLbmFb0EsLsssIOe0Tj3jx6adFxlZquMY5TF3kAcbCsTHN4zyowcr/ttH+ytrdkP0QlZYLgU5MsBHM90+E9SHqfAvwW9C3tFIYABSW5OxPFwhoG/bh/NHbYjI+E1GCuDYOf64MzIIvjCdGZlCsRSz+cFJELPRALzqEwBRI2wxiK1OnTTEQkiaJ5gGs1nwmA+LzltcHhjWRzNhz1cJegZs5QqsLe99V51GrBhkbnqIFZpLSuLpDlzWen5vTV72Rl5UWpRzrzIh0fgyxTqXIjlKxbvXIjoSxX0XMjmaxX5XAjnqxX+XIin/WKgCzt83QVCF88cV100dIoGZGh8tqRyaTxGNoo08z4bBS94mmHDKcOhujW2cvexysLhKi1KyyZnjpQ24JnjhL8rUKg7cooK/6DhStcJCQYFPHN4ytsN2fiY5xQ9iDGuMBo2PrHKd6H1o1fs0on6qGgi1J0cHKrDWpBcUnf0hm4uukw10hu8ufCuy5lXEGSV2lRBHs0pW0Un/9Z0sbPDusvByjudclUI4ymbYlxC67hf0wVNH72m49xpGEudnBGO8gGZREEjnAAPGGWTQRXPMs37OBPf0YAdOxG7WbUub/Dy8GZWwLzhy+O7njDvohgX1UD+7sGYZIY6sTIB6iGKjeRFkKFLZeoMwttJa1UjxCq9f0WJNOf8KyxHW/P9nRWXgLytUeQTHDPTPbMzQhCyZYeOi1QOxCabMsNrsForKuTBlZYensB1Pb4HFHkshfZEw0QDqU4DL9bZbvSW0EuVthu/Jfymo+Sj6hwTkfMumhbzXnsZurZRtUNxoYKdgfpwNxTMY1dcXzyMqq7srUOyRowNhz8U5ValWa0okeYMa4WFZ2umtYvgghyuy/lBuYuLe0vA+ciMWu8H5Y42c9t7Dfx0b0kMHPIFOKYNM4YK4QoB/tqYdgrFFl6VEhyY6eJQN+FOQ/HcHqlI8hraK8UFE1dftwkxChOqMOSRItPHfmFfHb0Z21TqsHGPLikXn0V3xDYzCBNHIz/wAJdEN2f3yhu7PLur8oSWBFmjylaSR2sKW9kJrDF17eyw7rKw5ntCQ40Chu0sI73DhJvZHO+NYoD6a3AzcqouV3gOFDcMhiGKE55A/QjnOsDJkg/5kYGmC0qfCN8sZ+iN3yS/mf7QG8FJgtPGtTQLOQpAF3klzYTIqJMqLtYGbpWogAyXm2DChdtQPCxIS60Q4tX4lUuDrNLWVphJc9a24jK1NXvbeYHBRBbGQe9KZkE407uSpgSKJDFciRWkK3fk5mwovpGSRi028HllAwLxtiA2FMYogP27aPtrQfuHavxKnDuoqH20FZU+DcTWvcsl2TXvXC4J7wp8yyXxXVnuQUGSdSYfFATSXvZBURnQXPrBJWhBhtblXKRp6SNpVioIFenXm4JRuaf92La9j5QLLQ9VD8NXLE5t0nDNGAyFLv0u0z7U1y94u3jPg9NQPKuTL7YlSSqiPj7s4KhSKvSj9o4qfIh3q3dbHbjzuviumVzGK39jN5PddTlHy3Gs0jdaDkdzrtGS835rntFzs7rLsTroEqg92lZxK1ESUHat4f3OLtLga742l6LdXmiCCKkZcxpsS9xR82qRpVNWuOUNdhDMCoJZaihsqqgK5z4gsC9G5two8s9UiLB1l3xJds3740vCuwJnfEl8VVrQSgJpzoJWduZqzYJ2PlpiovKcOFjy9NQ5BlEEh9QrCMjuMgzkmPguw0CJvTNmKvTPAe3hjlOkccL1CXAGByjD03hZNGyp1wb/doA6MM6a4jQI51jPZmYXJPPUcXYMKfU4uaM6crPMZzd0OXQz7Wc3eDl4V5ZdUIxjnckFxXC0l1tQcOZvLrXg/Kwgw+pTH2dq9mFR4Iq/9Ggw+RjbdDWa1NmWfBWmNg5TPuqq47Vnv7UhjsBor70cRKmFYY52BHW6/yRz1hlmleQCyVLYrljBGCLYk0Bs3MVZDlzrLs5y5Np3cZZjd10aWjmOVWpo5XA0p6GVnPdb09DOzeoux+qQr+6UdZlEqNqpTqmVir18PnGAqagMJLlAhQycCDlDpCHXAA5NMkPqPtUC8ArBIX6L18qE9xKPEcoTIYzzBjJzcKZuR+aUDV2a6thiCUxqK9gh1bYkuiRt4MZuNrtM1sCN3mx6DYd2FKRWpXuzII/mvJtF5/rWnJvfB+tl/f5tu36+fxqelfvNdvu6feseXtfbt839/Z/vOBv+6LqO01D/bDJKp5S5k1EaR7VkMqIgO2uDkmlLdtP6DgqL9rDM+G7j+gQorlL7YbRoxnsvYcXGb+r9uYLKyI22tpT9ihtHAg/gnZnJvO2tMxs252qxpzbXxA7LzcqED6CnwXDI89F2814zJ383mGAD6DPTaGqrwe/oXxz1l7nVkfJGdzpV5zITBBvv00oNZ67MvEq1ujkPLFW9xQ4fCLavrHgBFnO2vf9CMFLZGTW1QHaGsbNLZSdJaj62BJ+581ez3+bhvot918UnEpGzVNax2A1Gc156ai/lP+znMnqX7IxamWShJx0IFsVGcRQ22h3SEfB5FcIpafp//QedwrWOsI5K2yiQpCMq4BZwPFrLUBlB0MuBfWJ8Ly4nLkchFCNfqedVKgiX634kM79S1xNF+Wvd99bUg8uxaU57OCMa2KOBts0OimtnNRVqELYfjExJZ3E5irfP0F6SfaATSM+TGQDDlJbLIS3bJy1s/nYG7SxWCSacHal6xZGd22gRx6Jwg4u8JIWIOQAeOzUQGbP6QPTIBVFVmi90qe43b/C4HKr27SGXZNWaPnQ5Ng0m5J0YTei4YhNlrdjh8nK5fHYZpcb6kOXwUZ0oCxx7NLjBG5FYYRwyGBl++OCWEONE+omtG1HGcmbwaaWz4EMIOViukJxyqHNZfCju4mpXSp2K47QilcaRplWriFks7SLUvW4F44YUmV5BrG4xZhSTLqhrxYWllw9tLVKaYpU7zhZmEmkRuQkNEqeWDid66hpYN24uhqEVU4dTBUrTiBWFL44jXYx8+fHRmHJUGlhzW1pfihdkeH0aUpypajw3PjFcoPiNC6PxaVwUFdTvLM92ccS7/8vpmPgS7ZSIyygrpTdEMS09LtI+lYC+QkMITJwKWMZekol4iTo8EWmdKZWEa7OwkcHEBZaqXinipFqOf7cfOZxyBXcfrZFfHIB9uipTR9mauAbUPgNxUR+7JAL7Bm8+vM9yvaLYxkW5Xipox+d6CXyGw2upj2WVltDSz2ZrZqyyvJozbV0C110O18EciIxNBrsZHDr/lJkwjd91fKXi/HwBeqVGd6GPX1JKCLsftP2OV8gB9CF3oCZXKX4xXg6OW/9QCJQoYIXUxkm880qTxhuRPBnIGueu0kwmVwHRIpmGlkz9pjr1m0IYu+bdG+EDbjvOo5X7YO7TXOnRvLnLQpfSBg7ICjlGym2U90APhAoZ+IJr4RDx82PiG2JxihdOci5R/5jPwx8Hf/fR3wYxSR7kalZIL1Zvb/gW4UtXpiIKaPDxHemiAdJ6ERAtQrEroV9E2E7ByqTZB0KC4zy01huOavJ+nWxqHX6NaXClgbWmwp2bl4G8adJ8n3Ny5p6lsdvxmC1LEZTSu6d+av8w/En8Fk5eXYfv6/UEwAti3vJnHOuD3iDcAFuejOO0czJTviapN0XbPyanTLTTbhe6c315QQMrOb5Jwxa6qKD0LkkplOHBYK0RXpU+ydJQMkbddE0lI7uOn8mSBA2dDq6c8iGT4DlSPsAk2fkUOmHUh23NVErwcHCYDKMoPD7Lxs1N4MvEF6TuCb4yLJC4cnPH+qWDRUz7YsG1jsEZMWM3hIcQtuYtLU2sOXfppYBBDthSf2mmOInbxyz55kVoEokQOyQUbQQYfnBXhAn7HO3awLgAsIcqmVvQRli8bNpdhbNhwe00SnpJafNMWk4kw02w8Z36ZCwzRXFSNTjjm7FR7RxvnY0R8ri6odD5uoVKJq5azmgXmlDPqZJg7EyNEkb7riTGEJs5FTmt/HOVqNMOn92MS1BZ/GckhbzI4LTdEz9UCaw8wsSnemO4mOF1uVbL86zSvVrDo9qai7U8s+bcrBdCdpdFdtjVmon1UimqTAXgjKuVxRMc8luZ4PDWA2akGaUb2N7ahBrMKL51Ql3GOY9KeALeIjr8JymTU+Dfgt6F3aLowoAmN6djOakvZ8ahSBXhzK5kJnFNZPMIVHyf+jwCFkVHcewulS3dfVrVCrJKu11xKjOyCdKNaHRoWvEe7GTkZUCqHg8ZU8LNDvz8pvEFA4dq35ZHNifl4MYsZDYvL+FGLR5prZnjiiNrzh53XmJ8YoMffniLrasrxrmQzJct0LmQ0xcs2rmQ0FUU8lzY5y9X3HMhn69X8HMxoGsvAroQyNUXBp3iARkecLkdkY/Zw50ZS6zGIi5HCn+W9pR25JIV/oFjCrU/CUYBMIen+qDc8THbJXsQZFRAM8mfnL8/bZq3yiJZpXRnVzLJrCD/DY4zMTqGWl8g94+orBJfXIbyxm8Zv7SW5YkJHuVBRG7GuEM78BVGeF3uw7Isq1TWS89wraliZXk1p6ldAtddDtdBd2FTYf1ISzFplcahCbzfNRbXVPiNDD/cPwB3MUd2Oo7LwvqjOJ18/E0UKObD+qMsRKAC+kFUnhgq6OOLI5ndWzadA3doH9biIKt0FZaGElt0MvH9PDy8AAgOl891ZZFMUB1OcOmeD9gbYOHOmmSuUMK60O5XI77E4HPjt4jfvAyTG8HPRmBr3sTSxJpzJp4bmIC8iVIcSizMbdYaJt74Su4uzc/JCM/jTHLMOKH2/rsJlViD1VpRvRKutPQkBeoutHylwCgjNO/rso3dslKdjGTDftnS6GZlRdRc5qs0wDgpIt0bPaeypoUlTJgn7I53TdHOo3Jk0asUYJIScSO4kGCVVrjyw6o1O1xpYs1Z4i4FDHLALulUjR5RVOB4nERICpyEcWy6GXyqTu9raE1oIY5plPSGSgAL0T8XKHkVTsV4UUpwYLSbdZQE4U7G8RqcWoURXpVTqzDLyMUv4v3k84XqXJo7khuLY0GaM85XRyP22H9xHDXqWsVHSGOaVmFerelZl8B1l8V1VR5P+pxkoLhhMNwAlJAUt4rCEQCV1SEBM9BWQenTkVwUpT9vw8p0mudh6im+DfVVlZTIRH3VUN27ffMoBTNKh9uSV4myyjV3eSwzCnNFosFkMxl0RstYmrgGVGtlv+IS1UKbU4vrRi2mNiuV5ghu0SGJW1p0CSkKPa4f7eUEUSQ30k7m6IrHXmtmsvLMmjOUnR0ZTGRjXNbTmfgEvOIxdkANioe14/oYvcWJ4Tkd6DbZ4D8czYgOv8OZIRgCn3hBW5Ci3qRR7DPo4v2z9MlAZrx1GW/JzI1G4wL52QB6sRJJVTlyl6CGP1ZhfIolan5a7AvEmToZzqhiJqPDA0y2dM98MtDZePp6hTjmVCT7SjyuLD2gLMw68wNKP3DNJQiUBdZehsCFeEGG16duyTSOh2VKDSSWsKz6cFxcmVPG7SvYTPnUOCoWGt9ANWhg0A4FThna0G3y19nFGyicDGPqlYxMYjbvWySAGUdG6vvgaYULMGlBNkSIatWYfj+NcoFafajk1YbwurySRVHW5pQsCqM6n2RRGlW6JAuPj9Y8kkVxNeeQvACtuxytg/7I2qPNFQWJWUP5q4ZL/3XS4GtUGcEpGt1eCwnAGXMychfxP4o4dkyq6NywRYlDwR9sQ4d6A/6PDOLc64NY5Zq6MJOz7gcU1wnmSV5Dp1aggky30VY2+IYDWzmWpnfevYCuHt88R+QN4CfjrzWLWGFgzVnEzspLTBSDEweLk2bj9OPcZgrTjyq+k0Es8YplDGIcNb3xMdggjHViFFc2WMSY5i6sMJELxXNA27+jrmNweHo5bwxyZ3gaO0B3g/pt8G8HqBzjZYiTcZxR+0xlNGAHSQXSjLabTpqc8I1LZfTWMK0UjzK/OA5c58xeTamR3pzSZzd8k/jmJYncAE4CvLIUkZIo60wCKDu5NZcDUBJXeykAF6EFGVpL8yxn2v0zDs2IJA67eLOf3h+HPRY88gNyzgwg0vEX5jhSgBSqelbi9Rp/mxlts8yskjhiwVCagVjBmCPYU3G8LglQFGVt7riiMKpzxxWlkSqqX5tHlcpS4eelNWWpKK7mlKUL0LrL0TrgZGuurBiyYwqZk81Q98FQgNcN+C2SduVmwrt5xxzl6Tg27OYtC+7MJcVinZ/blUmlCSXi4kSx96XIoaYY6H1+hK6S31kTFI7yJnFr8cc+Fj514jtvQsP186syvqD0kGrNQVmWV3P+ye/D9f/Gp71JgI0BAA==', 'sha256': '58135004b4ca522653437430ce6132d76e0bb90b3b3db82b2c05e1373615e895', 'bytes': 101760, 'original': 'issue6_summary_by_method(1).csv', 'original_sha256': '58135004b4ca522653437430ce6132d76e0bb90b3b3db82b2c05e1373615e895', 'kind': 'unchanged_saved_output'}, 'reference/prior_calibration_quality.csv': {'data': 'H4sIAAAAAAACA+2b2XJbxxVF3/Mrhlw9D6/5ERREwhLKFMmAVFz++6zdpyGQxIWiwDJipSKVKKLRt4cz7jPcm4f7293z7uF+vbtdPW73z5/37zfj82b/afXr5vFxs7rd3j1vVjcPm/3Tdv1+d/+0ul8/b55+1f83d5+fnrf7p9X20+Nuv7vZ3K139wzc88vm5ubzfnPz++rTdnO/3m9/2e639zfb9eN+97C3wc9P29uXn8ev66ePu1+ebWDz/unV4M3D/S+727HMp93956fjJu/3uy2zbh7229Xdw4f13cMTp7rZerfe/uMzx/ltd/v8cbW53Tw+7/65XY+vXn0cdxtHWh0P+3LZ4+iXDY5Dp1sdv3uz6fGL89cZRH+1u4182dk+nu5q4292tMHzuz0+wLX19ul592nzPI41pcLmHQ/86kTf/tT32Pn0qt/+7Beq3ezW4ef8+hqHsfOHPMw4PcLhm5cb9Lqwgw1+dQubsriHffVlkxcasb7d7bc3uvVLDVjffNzcf+C/uw3TD8+9Hj3Z6PXXj/uHx+NyD/fSdHb527QD67+/Q67sw2q18u5nt0o9r6IrK/dzS6//aCjW6JwrOSTnem/LQ278e8cPV31KsfbgUkypRM833ocxWpIP/K2RsViLi83VFnoOzbesFUoMzrUYXebB4ILGau2+1NxSY9/g8ip/9xXPndv94X9LJmN1/yD2I/73Hz7vnj5u3t9t17/sHz4dJf9PnPIwrPWnzf739UtRefrb0+94BBRyd7Ne//ouQJH1+vYn93N2q+NXqzAullf/sdD0WEJ1HV6UDvG7yNNbr64lGNNzLWdGfAsFfvSQQ/QpDOaX1LsrPhordSDvcvYxx+4iv/vTJ8N40gUXUwm5Bpide7+mMLlYG3un1nMpvdVxoJg4dqi5uBZC084uNA5Yomuldd/HJiF5X1upIRYO0WM8pZXL5/Z4PVGXXtzk5Ch2EZeKzyJFcb6UNNZzuXnHnUtMIQa7sm8tBZYrwXefatVELcVpAxSrubY8juizLywaG7NK7j4OvtRQI+zzcLGkNujFhJCTyz0ha02nDiEVGFJy0UUcV/64+/AREX5hn58/vgQt36Ilvz3IFn7DxMfN/nmH1cWGf9o+g5rWt9vH7T378uj2wx7nLYN+VpPcgia5CzTJdYjUU4YIHWoMKQzBl4a8ekls7SaZJWXH1NIrIjhYVzJTs3gQU8umTB7WNTjdPYLgxbhUvUshdQk1Mmfa1L1EPcHO8bAJlkPDIroXaruiZR6jPkuAcwkOIuScTdSRioQc+s5UH8ad0fOK/PtWmevT2JxfHGaIFVJCNdICEQ8asLjPG3q3c/ucnGcsyQz2TSHAXJRprFjKMFgZJYBJZW4eE1SD/zVmlCmWcc4IyyAYKlWDhyNjd/QziK0YAA6Cbo5jNjENFuXECmYNNMblOHvoyfnhn0vvKGQLDQXOrscfQa88ngm9evfWQzEu2l3golry0E4SX2L1ZmRdgiPiM4OIvTNOVx5AFFBCRDIe5DRg7iCyYzosqqYNGFWMFxYPg5uGssbeeaqjpaX2aPMcMuSSGJ3Q1hbMwPuK6QtSY5xWvaqv8k0ijFQgDxhwNzQJQ94L7gLFwQM3Mz0cDYksteCRZf2HjY/IElLlIRE3SkukzOX8Rm+mntvm9DyTD9JAOFFCk2Wa98SmJZ9TkCPryffpjTCTHID5KYjzdijcZMDbuIiNi/MEroQAzzmWBns0tQOINOxm4mi4s2hkDrkFHQvTWt0YS9jHGCOO1ZUBRr6fhtnPP6REPy0p0WUwz0ObUtGfgn8e4ulk1QrE4Dv57XBurALfQkDgERsfWh3inr28DfZT2iWupwhNUcReIW4fZhZYl7Cfeh5U0wZWeDvWrqo+ASVHiBEr9B6TPMSqBCQXiwtoyj4YluLGIp8kuQNuqqG9jJtFSDkT+CwsEfH8LkuUXdrl5DTjNhAGp4VVgjhgh3Eb0AM6iibzRWwH14T/glGiRpL3qKZNKEyV8qcBSseBgOvoBhqBf0p4XTtlg30xea4JbDeUi0FsQjYeTW15SBjPVkfAgHLGpEV+IM90gvhMqS5AfE1kwXhgLHuFtsOgwG6gWOo4FcD8WM0BHDBhjIDDsaCDJyAIAB5gASOH2eoT8xUQAVJBLBG7FoQzRAM4KoRBv5hT6gz2jExgEZMhLIwykimcniMg6Nqgj8txBGSvVUKdoRpQBtwl+aqgIihn/oKH8QDgKcUpQxCBtwgyk/BNhnzekjF8ZZs3U89scnqYsWIGL3gWLOglmjcunpBo/YAr2CfzppxNBCY6wuAB1rv5I4JmKAO9oR8yUG0qYR3Arng3mNFNAkBwcLYKmKTpj1gcjkFJ/iW7TBUHnMA9ckLE9ZfxRyEvg7pwMaqD48hbkj3Dn5v1wpMTQ4pvqAy/HFhHbBUHyC+Kt20qcgBKJz4FNCRLZwnTgOtgQRBokbSjYlWYA0uMTJg5BMApACCakirkGdVyNAW0rQi7xKu6paSoGUDUWYTjj8fZOoJMAUReWRQ/cxCE3VyTkwpbmccQgaAvhODbKtk8JWWqyxstED2f3ej0SONKLmONakR9lLXAs9uqAOcgx9MUGM0FsHWaB5zUDsZeIUCFXmAHPEzATw4Ep8GCDgHNakxhmgA5QNQmg0fA9MVWKEpf4VIrp5L/HBfA+0ahU0Ad1CEUvHv47S+jSD8tKtJlyM7hiisk6hi37CZB8C7KL4gDzuLLxTGMpJQrR+UJpg9KygSWJHCWzVIpFJehk39Cfi0zhFKViqUEy8WpVrK/WFNAAuzMVw6MCtYhKMuGjDYjA8KK6AW5lcTvFtlIU7JSYUJKvpjX5mCKDLHHKEjKCwQz17C4zRvK+rO7nJ7HsB0qEoQMcPEgN6MG5GIEsI4HSxMEZq8EIvQBX2Ou2tQemFDZAaUEdrduStUlKI1zSnfaBHIJImpJECbCZWOibwOu4i19NwDLl3K1WFxQqW/hxwB3B8VyS4p1CborGaYoo41/KfFgfxkCbcv1i7JjEEDPgxkX0xTc2kxAioJXWWTlYC1q8oqeiZk02Y10OFGToT5leWGaaVJMA/HjjOIBL8LeCIIhxBYg7+3KAM/HqJSiKIKu9BlBKAE20gleGWIDeI6liN4b9FRcb4Yfq4/4E4EHJTBDXSAmJvr8Vm/mfm2rk0NZxg69hQ0+S3cIbsaiVcrXhZ09/n/OdMqrO2UdlFrl0JMEqQWFRehSJXCLEwHiyHojMISIMGukaXkamxy86iBE3c4iK0XeAS7C/Z5mKl1IkZthuwnFwC71u2jaf71I5S21voAW3YVgsaIeCanGSvVwAB4Fh09I5QAJinbbamlsTJQO49+APJB/QEW5ygyYT3AfRU0j/4chxYkiVXA+WR4Ay9il2yhSEr4yMyo7j7xgdhGSq7o5Qj8PmlHswERL6rUqpIZtryxeZ2Uqq9SkZBlOpR5y5hgQZuKXRBL8RlqiY13c6NzEk43OnYgwD+CJp0uyiVB+bi7tLSpXocIxTJjapJnoVcdctC8U0UNZ5SyxBwhvOFG7YBBRWHHSmVOFN1G4FPfF4Cy8oLbmuLGh009CYFkLQdwo1v1blPjNbuyPoMRwRoHCxQrU4BzoHzcG6YsFqiAL3D5agSCWmaXKckRAftxSminS6EeKtykrMcqKknYHZxDKMiS+j+JUKsgHXI/MygaX2FshRyNsS76YJVfyC4eqLBW8QQSv68eqMF4QXoLzrDuDbsJ8VXqQ9zTzBXwcJWqEnyDIEhVVlCXuUBY9zHzdGyq2r+yzRPClfU7OM0u+CHLG/vmsIGwiyqB8rqpbqqYP5WnydILhwBcQfDloZMpZUReqFWbiH+3IUWUy1RTxkJb8yAKOYHlMXPWTwIAZP3L50scBZUR9p5A6JKHhv06iwp3J8V1Y1f1/U83/flMN+GIxu6XxSwELvqyBVzJBlf5oGWWYvPowoDvThtU8HXs3lAuwSDzOt0l6a20g8ExdGATXCshlcoPKEZiEJAA8Cl/Y2dwBSMBUeN7tyderXbXiQhgkc8LxilKshtsxOtiMUUGqjM5kaNQ1Iv8R5xi2yAqYiUXxM2qPiEtEDMu7LMxMX9nm5DjKrFdNJBDhimXujT1XrBwUz3QrSiqYywqxcWmxWUSoHgTi9KzCZevHbATxtJLZrJnMIUHQqA4AaAekbNFEQg09VdFLzKpyDhxaRvTO09xHuPbbcMrLnzd3283+pelcv394/vgtivHTomJchkMykHn0OMQMCrNar6iC+BQEl6E0ydqLJCmq7CHgbwlIlKoEBFsJlwFP4BrUVzIGlxmtAUa5QJVPqrjI1ANql6eMYmu1jWsD70srEA7g33UTVso4c0dVz5R6OhT6kGJV39SYAkyfJQUv+VWazzegweziUnrHx1Er5A9BbT8lpoWeC3udkj2G5Z3OnGrWGpFFL6CPRDs/c7xE1yB6qVPIskojGzzyvaqUJdU3LZeLcjhlBZitLIlF1GXQiKhcgQJo1fJzSRnI3lSEhM/TjLBtJ/ruKmz7qWWIS8sSuc6zsCD9b4TUR010S5p4AbCpPap3UGl0WDrTJLJhUHlgz27myCXkRQ2DWGGUKc22DXwMrBTGzPB5QHulsVJXB0lSpXg4HyZhIJEOOS0k0IrMrN2UsYSdUrQxBtuFLysQWhH4lTtARyIA69JDDVMQR7U8YsRRMxFgyCGwnSgWH4MhQs5NEQW4ldhS0b/5ukTJuLzPOZov77N0IlV/2F+Ja4JrFpppLIvZoFxROcxN8wJBe89KTAtGqFrwzpplCOsjm/qKczOSKq0tz8ca3FCRyVhWhhVswxIKwi2zVfV5fJWbZVsRszYCFmQDonzHsPpPSyC7M31r7uK+tYLNTV5MU+uv0VpIBSJJlaJKbFrtdEwTlWMibga3NUho7YAJ5zmcmnLTwzJmZYO9UhteydWxXIxB2EEJRVg3cM/b1a6qXogx16usjK0oaTY1K5RlHnpj3XuIntNF9FHOfEwTYfiLpKIZc5MlEp7ucToxp3O7nB6GYBj8FeVBk0yfscQ3D8bwKpZWubFBQgxXUVsbmoUCWSnZ4856FmRzadZzcLPq4GGuun+9raem+Cjsp6zw4JNTU1RSohLiRzseVyDAxngSasuCXgn6LXeguUs70JqQhqxEAXzHQ5G644Kckhhe0NsyRHh/pcSVMRKCsJlwUSlgP7KPrDsdjsoCLY5mdN+s4F8HglATturW1sWJhKutvasxbaZOQC4VLI92gjX8lcFfUnUoCeare8iZ0URpES+ZfA5Qujs00UgIlVRTwtbCHXYNWlaJaegUF2hp7ndppyW6h3NbnZ7J0lsFn4XKIalqRZ+J5Ki2JVUsmzoEyvQ5KEpP+kLHdpMsuBB1GyhEAhbOhhg/WhnUvK4GeuWzxvNok5JeSILaOy3PTxwUDBWODs5u5weMEoIB75Myat+j4v8XAH7LTWvu0qa1qvATqRwgWdbp3RB7wWhl/NThFyywkosZmgg/m7kviK5WdoFrdZTOFIbePVAmGDM9X+hRN29UUwfsrYQMMzMB+u8s2dSsMh5NervBK08vKJqv2nVDlK1+fVVsVQQaIpSUUSAoJPKXr52N+fhWdaIgb3oPwEr+hIxJbStVpdu2QMTZ2/Z2kwVqp3O7LJ2G++pNG9DeQNXFFqwqmGACCWZDmUaiAgiaIGDVG1723gPBMsi7YiuVL24Tq6lvKhd18cg42mXcAJkosRfWSNYaElX6ArX34R2tYRhFj2q/U2Cm5oU/yTsd35PMx9ck8yVvScqvNJUuukyNGUmV9qqHJFW5pNnydzpqbezKrIO6lIuTRzaJVcY+qS9e9RVvSYg+ujrKsVxcpAyj80NFE0MGbfT6EQATNPHPX7ceQrycdJWaR/sVMcRh3KnHRK+sNTXLWSHAqW9BjSaSDiQwx1NCOSvLvaFezmf3WyB1ObPZuaOZT1IboFX21Pzp5yJBnTDoag7jpbfD+3O4KHxPU+ZOjTqzwSar2awi9SjCbJMeLWcR7uaRuwrzhb6iOgoAAhqDI2c3AJrVBfn0rg8kNl9fuRdKp35wwGTOP2pC4qiB4aiB4RINhHcKjRBO/WwzMh2E8urGHch5aMGo6hItEwx3tWXZG40jmsGm6ufIHsgnqdqcxwuM5oFGqweqoHfr/OzyajLcyszq5zSTb8fCNZPjblQJRieRM1PCcUdPrNByjWUK/YgTVC1VqzO+cgbjo5ytPhUtUmJeouRMXJxstTB1Fi9PN1s6limBmmJgQ09qypx4r0MNlRmZyGXnewmgNKfXQ/QOEmQ+vO2Hceyjrz01e+VURUk8EqCwKCM/U52KZVE/r7ezlKKYjT4N8B4AHdF6DFSV7upE1dtd+LX/yBd9g7pcEEb5M+9r+0vf1xbc1otw6s+QqZw4ATSNW4IRB+u5OBYkm11vrWX4OGOoqIBMPU0CHq6OGCorciIUzWKZBdhZr22p8p0JsM33d5lxZankB4lWr5s/537Eb2W0+cZD10rCADfuoVdniGzm84KrWV39teY688RR71oqW0n812cH62uizTfhljZ6Q91Uzm905kx5vMjLrKCm6jIzreqqTmqtZaXpiiE7rMrW3GuFRKc3jdWZBGyWZzEfCtjGFjhFdkoG1dnOMdrLBFbwV6ZgeeB/dRagcnm2MCbO49VdqHe/24/qo/4FXqfOZIBHAAA=', 'sha256': 'e426dea843790d0dc5a363ac09f5ef43ab4933ae6c594bc3e7e59317cb85dff2', 'bytes': 18304, 'original': 'issue6_prior_calibration_quality.csv', 'original_sha256': 'e426dea843790d0dc5a363ac09f5ef43ab4933ae6c594bc3e7e59317cb85dff2', 'kind': 'unchanged_saved_output'}, 'data/coupled_policy_states.csv': {'data': 'H4sIAAAAAAACA62dy7JkV3GG534MRhBxpFj3ywt46JHnClk0piOghVsNDr+9/y8zd7Vqy1hwaknGEnWas7PWzpXXP//8+OmnL99/+uHDdx9///aXzx9//Pz2/X/89OXDpy8ff/z03Q8//vTl7Y8/fP7u848/fvnu+x/48O3zhx++PH/w9d+/+/3HP/zhw+cP+oX2v/v9h798+aP/L/xff/jxz3/WH/v04+8/6Lf/9dOXp09+8Uviz/zh4+ef9Bs+/u3D5/+0j//js2T+43d/+V6/0n/I00Ie/8AeGp9cv/CLft+f//KnD18+2P80fvvf+al9gb99/6e/fvAv8LN//fPHT3/96bvHz/+lJP31lr7tXf+vvP3XXz98/p/v0uOf//r9n3768Fbe8lvVf37z2/T2u9/ED7Md9/cfP721t3//rN+fvs2j171zb7PuMdvmo5XXzjXVtmap+rk+S3m0XbI+abOPPOq4idFflWPutEcqZaTURys8VM/T81cZQ//SS0GO2nNLK7WU9ZO5R3/IMY6cRqmlj9VW63WkVU2y3HPZay89uefFc1Laec2a2h5r7JxKexbj9dMYozf9X+p9zTl4LSWVrr9KTWn1WfpEjtK3/utYu5ex29omRv515Shv/R8SQ4+ZJe9adi5tmHLoaSvPlXarI04jj5zTlqRD4s0yb1L0XxUjP4nx6af//vBZ//JVjl2zvveuegWl9Gq/tSTpa05Jr0avaNlpzN71yWir7jZa7w9BjijH7LPmXOuUgqS2ih1Hbbz+vWpZ2b5uSnPMnfOQHPrZWDcpXtaNnaScueWq69pmLnZT6s5NckmWolthb6XUpptahl5IlVa7ipZjhqPvPXdLuh26odOvik5j7jl7HiNJfV2KnpLuqo5lzjXGTYr/XzfS27/9+OnD29vb9TFvoTZZoF17Wb21aTejFx01J1KlFPblq1RU4s3VmizHzo/HHtEE3YreJcfihozRXK60d5GhGlt6YtqRsv5QXTKsuhdS0jmfxXjdTOSFMdhTX3EuewclNdnNipWQrRx+QWuRJZld7yH1ulI1MeoxVVitSIxtx1+WvYC8i6xSa7vLUve07WLIrurP6fCSBM613cR4/WYsvX6/oLv4K8BclJFnyjKRcxQ3E5jNKi8zpEV1fZXjiHLINPJGZBETSjjCpaa69NfWm1l2Z/XIVVrW1ZGyFulzr89yvH4c8h9bvzhlvHcPQ5HKajPJUktJ3KfK2+Qlqzp0m1eV+poc7ZgXkdIpamhb31wXsnQ7kFH0TmS+JMlMbr3l1fRi5tRr3KXfpbgfR/bf71KUt9/+7hc/Xz97JZgh3cqdxpCR8uBCVlSXRO5D93WaCLUVBSFyIQpGJHR7iHAouJD9U6iF9ZI5CF2Rvsp5yLPob/emCm90EkuR19yKcMqzFP3xhOufv2oxdd+6TLVUXn/Ht5dDlTvVfdX1zNm/vA5izdSX3JmCDntsP2YkFGCmIXOoIGHr25rNXMSSirWKfBfezL6+rFqSzrbFrdWbu8nx8rVQ4EZcNRtRU5t+HLqxXR4b9yEzabGVjJmUpsm+6sL2UR9ynAkmOGe5D0W3chpl2Wdtb9lnaak8WhgJObZc5E+lmlM/uEnx+mE0uY+6Lcybwx2ILLnsuXy2coBp6pLQkkLArRszsqJwE2MczEKk/rnMrGh6S0lcOfTZlJPVHZWBNDEIsmWsJpZD3n3cxHhHNKHL5t6iEUjaoSrQ1AEUxdMzK+b3vCMX5JFqEuetrw8+Yxn0KxU1ySoonRgzlEFpltyktE+3YfndkI3EXDaZEt2R0MlxLp7oSdGKDJWMo7Qz4okqhZAC6ojk2Ow1yLnrvkgcvRmZTpNinlMGwua6BtJkd586fn2Su9JAeTI7CnnXvaUDs8mR6RrfhHiHKgz9Or2ERTSlv91M4rbkpOWTyhVaKqpZnUOQ37iih3lMFZKdP6G8AhZPwhTTyxoqlJrLP8Oj67oMkuGi+LvvZyFeVwRFDlN3AMXTI3vkn8pwdPzyE/qp2wVpCUE/aakC3+0B7jqXgEoABVJDQbaCpRIZqGyRfJrejN5Kcm0gBxxyMTIhMu173OT457VBrkmWWEmMHHbZ9qt4K5NYTTGsLl/kGUqzypIGkuDoGB4PPqMNW2fdBmmnznd6PUL3YLSh0E5aUWv1eoQOYpMDNJ3SSDcxXncTFEA6102WMZIuPlCwpOxvr8j15NnJb6oiF0W8JsM+F0ZuHQO5le58r+4i5JBkjHUTiuTwm7H07eUnlQ/rvbWbDO/RAx0yEUvvA19hOYW0fVJtkF2o7qepU8lvyVEqhlWk2R8PPqMHOmEFjvIRs9WSvWCmKEaqoJAO6+BfPpHkkfVKZWRJxrMUr6sBJyEfqGRSxneFhZQODqVZCtpLcaugNKKTYSiu1kf2EnI6pwgKh3CB+GNZKRdMkfWWIdTX3snTmpJJfni0AicZ8ZsU74kVMDo6bgXM+r32WxSbVYVyiokUq1fzmEnGQLdEMQWVRAVMjwcfihX0AuSBdkHPp9ceKBNuWSddguWaIDc2GzdTPnroij4LcaBAqZBxKJHUgxQnukXQ8fDCl4JaZdcRRDeZo9mV+OvSKsE0OfK50gNpkgKRLNPQqmUQ+hcFjvKeSzZcjsqLckpBKRvPKU3h2tzkePVA5B2V3s5S8UFKrb32MGQOJZ4MljyIZfZJP9uZOoUCnKV4sj4E+f+0I5sY/6d9IlRNu+GcFZHU+2fUh96+Wd9KCyyyoD4rxfjwTW7PT3693LCVweVNKahLCdw6UJdremhXFNk8hm2bw+IOK6zrXjrP5+qSk0ZFwhoobog0U+e9cR9bQaPE8wuiDFNWspBuS3XuYrxuK8m4FaJxKgqTLM3UeUgZqsL9ZGbJa3MkeZPchnjyqxxn0kyppTJ9KrISxaNpXYSEBcFk7OZxDNGbgm0UUtmdfvgsxoGkW0c8i4L6TILpwTWhQtKHemVkE3Y9CvmeVEappyy516JyPec7Rq7yXIqoFaRkD7CXcp0xZLpkLsrwYlRWJDNoPCnsVCKYb2K8I46QlaK6RKtEX8+rDkr9G30mq6WHOlQ6PEN/YlETbI8Hn3Eegyh+yz1TZOmeaE4zk+iE/LWXoBRWyL4rE5vKQaWiz1K8rg2Jgj0FWLmr7IVaGUgst24GdVn3pDJViqmJpwaZpr+Fc4VJWWVFR2UrndWJexl/681L8YZyQHysK0ORfupjXY1dWjQ/cztYuG7Vkj55DlmubKZCTmPtTFuhllAN+VJ5cnIPxfgzfOmx8qTcQrH30nQbItXKdNhkpRViKiN1T6qsS7qKn1tSoZH6sxivH4ZS+9Hkz5Xtymd4J5i0X2Z0SEelul4L6kvmHFEltoe6uR8sQsg+LvoolQq9F62Lgh1S3r7xrq4bE40lIpQ9qbvexHhPlKmLkJV1K0zpqUSQXfVlMdjbChTuN5ToLrnVSnHGKwC5n6tV54o2KvVOO7RhUrce+j8ZzBG9TnnZTNat4G/Q/XqW4kQfvFKSl5eWaZoeZioVlueWjiim0yNNGQaNN8WfZcpnOCognytPYgq7AicpocKWHPWBQeyrYEYq4MFuWvYHZOeb2ax8E+PA1VACOAmb9EXNeelg9IKUiFOmHuFDx1aISV4irSHOfIhxxk7obvKX0s+qA3DVQPlz53Xk7KkR4YMuaevKzRR1S/byLMeBCCvJkdPXVC4YhfykPFwGEjjLpO/hV5SkWbqiOEipu1/Rc+VKoBBoiJKxxRty0Yh4AS1k8h+v0Sz6CE32U8kRlaSbHAe8iGI8eXcAI8MvhiLOTWt4KLqrq7sFl+rgegeAnkIz9CHIGf3QnU04corarUfI2WVRdCZDarC8GwyqYw7ADDoV+bibGAdwElTHitUFqap7aZ8uY6V2lYiJvWKkBGUnEiPd7e2GdJ2zHXgpqagibAW2gZNY2DMFFOBGRg9IEYnzpA3Xttx+vcnx+nmUIts5CpoQKAR9oAsjg6r4T2HojOKNoj7F6bJuSfH6fMhxCERD/3vrGb2OEca1KezRVZEDcXCPpNOdnlh4eTvFwfVZiNdNx8KplWERTLPwCg+WrQ8E8CvubF6D3ERq3KVIAaLJB2ualaKqAq5E68tSDww6+mmvpqxov8kT41LoB+9a1k2Md0QZUrElM6A8EODQ8HyMqpriTyIwr6aSpivo0BsCW+Vl5XyuqqksbMqBTvl4gCluPSpWqdSsezm8YqBcBPWQh5Ol1fOfpXg9yFDuI51UxiXTNK5aVsVK7wR+x18Cgbhu8KDYR05tulDSwSCjUYootFjkJBwlIU1VxqE8veDM3Y1MOg2K0/WpzNq4iXEgyJiyjeY/h4P+5EQooCkElUkI861XR2awsyy4wq79kOJQjGHdPqUd8p/SeLccSzEfzaa+VktuNIm1lCUlCk0KeVp9luNAjCG1qNxOGa6WPMaYwHVAQeZUHFdjoI1eFtmSElZpksmRD1a1BoG1YmCZgtm9bjHlMYiPARKN7JkqfTL6UAN/f5fiHWWLpVAWUIhcWN8RchpGRheSqmYbYSjM4e8x6Ei2x3PPaIMcBogQSjZyB93txCS63ZXKUS41irxNLquTLhRd1vIsxuvKIEeQC91PnbOnOVIGskOgPb1sR/UksKmFNtVUtEcIaHKcK3EucjPgpdI5kGx+azHbJGxkiS26gktx3xwAqhvae5PjQMQpR0rUomxZr8e0gzbYoMhNOyZ7PQmopvKiQaeeCvlXQQ5VLpBBtw5d2OtCfG3uAu2JFTFFk4+layfbooPxXL2cq3GSCoIcT53Kf+AHMm1r8kHlho6l0XuyjEHBX1/FGwDlHBhzUq6yoEoJYHJDoVupi6L4f1Cs2I7Z7tbfJ+zdYHZvYhzoFS6bGZAyKqaKXHVUepjS1extSZqVFACVsA6ADPkhxaFyJ2aTmpblYQHijsx1ozUReSstk1+bW/GYrNyzEAeqndP8eAfvFTUM6WAhElbqLhUI/OPAbvShP6cU0WK8chCFmclxlHg2xQ0rKt9KimXIFwYTlXHjtTIxjmSlpxYetb0Uayqqb0uBJqCK7qVvHTrNw4wZq8P7pqDY5bp2pwaWHDhfzsEva9PVmARxMgJRtyDhk8louiJ0SLxBNaUJRdej0mANY/V365v/DA6Vzpt+qVJuhU1R9qZvuyi3KcquASpJsiGNCiQooOYRXj+nCTKJclj4sD1n5KMyjtI7ov3IA2XO1lSIsUEZyIrehHgPkoIZEuue02Dw9rlegOIImj2U+boXNsfqvZpHlTV9PPeMEoDc11so4BhqhBJ28yULMIrmVV2lA8o8FfOmRn9mPktxAE/TGs56VgAmjq/SQQDypIyWyDs8ymYYCfBndziByXGwsDl0NxXEkv1L4UdAsxNm0MDqLfJx3YdkjXaJIwsxb3IcKNaQjYPe1z9b8wNZgEFp2ClBjdKmXAUNucKnspTjIceZQEKRS6fuLXH07HJN/cgUymAr24oAj+56Jhva28oC5VmMA91S/WZlO4m26dhhKBKt0QEyPfW4onQiKkMDibDPpDiIw9wkNHIEUjzdmhItEIbQ9I5kEnxATmqbZus5rWVwhwh3DxY2SW90aRUgALIL9P6QcW4zsh/3H4WymT5spKvpqxiHYok6JICM4U7RR9d7kj1J5EjV/Zqe2w2vmKjEyn6UZyEOxBIK6rkQUv/t2WixgmshAas1XLjZ8cbUhU4pAt2DJU2lgrKQOuXGDYhIgmSk0xPEuLkjzVi0Qcgtn1vXTYwD4HUgBYvOj5zKuhwKBZLBSMd0OWplVEs+N5ko/SHGKY+i9DODgUtEOp6t840rZmsPD6sYBBwKI4rsqm7TfBbiwBUpqNzCYCueiSBTPlRy5aL3FBMuygOZ0JvWIyC8MTnOFTSnIlpAJ2XSrZxR7TYnI0FkNFpEFxNAgQyXzAnFhZsYr5/HINvZjlnulvKUAtaH6kzjhCLYINBgUg6Pm8Of7HNDHoXKhU14pahvLpvFsVa7K4Yc7wZ1DzJGQfp8FuHAJdk8zFBZ3dNdhRpgY3U9Vzf77ukH092APWRdGFxEjnoOtdkNVgGEV8F1qy2Aagr7wBss5Iw0qEsvZTgmYJzhSNr6Em5zEFQB4VjZMdSGzqyTkaNNElyioS4rOmcFM1cc8lPPwTYBejE6AIjEJwB5lm5JplxHaXvHxQAXWLqCr0009CzGAWR/BUsBNm74wFmxhgzzkGDdfcJD6XFqCezz3lFUrefGykdpjNjT7WGm5MpEKQx1RXX0J92NMU7b5eszOKBxk+IdiqAwd9N3k99onnXryzdGL2mTgi9xo6CQgkE9BbuKfIqn4jWfU4WtQEJJTwXHnCLKZHBdBkifrwuF1eA9YP5CKiNbOp/FODDt0xm5YmJ4rIfPUMRHX9KADFc2CsyB2ROlqJcY56qZilUYuNHdYL4/XEbXaeg7y5HgXL3CrteRcOurckT9JsaB0ArwspJgoPXFh6mzYeOU/DBOkLefh+I+YBUSte3t9btazlUpGKgAylusRRnFf5BPcmmFOSxH+q9Opg5ykKPqz1K8rhsrw/IxGHRwFC2wTfkNA4raaFoA9LYhGnQO8rJxGAcny2l9Mtk/qORGCoIJU+BPzOOduTSBwWzKaQrFm1ds6snBcqVdGQPRgd14oGnohVzpwjlvCgPMOheAvzJt9M8fYhzCVehFE7wN+nEtmqV6uk6jSMK6vD9oPX6p5uCyyoC3ZzEOtMQYyMxguHUbs1sNAIHVKVF2lNEgJOEN0YGIllBtBzWjMVpAs2O74qMYWFHdWAX8LUBIk/Ep+Cf4aIUTOQrdLNTH6GEzExMNMjy3Agyl6znNq7rL1BilPXBL+asg78P9K5rCgWf7dqXfPoLyor59k79lNkfxL1CYBAT+wzehlifxmt2K52vILORygXnhnKB21XuE2nQctiVhlHiX08TUg2XNIavNiLkEIcCLjt2geUlBBQSOV00o/jNVXBRpjnYT4x2FTQVUegzZaCtXeUK+tY86AV2lFvElGAe8KLQDPtdezxU2wbHUaZ3qGDeQQjDCnRlf1VNjbjbTbqFPqMNiPORZjAM8TvLTzKZmCg+O3qVHuumFzq64IkbECMYLYR24b59krgcHyrfNaulyzqFArlxBBWQ9aEXcGZ2A1Z03YObF0PdNjncB/WWHiK0aUwSOnyDtadbx0BVtEValBtDLdXOmrydwqFwlRyVVs4b5WC2meklEu4Grxw4DOdHIBti3pJhWq+Mk1D9hHGgUu54VSuqMjo5u4MRrKooBFHwbeG6X4iBGszQbTBoQPQSRlSE0GWkEVRVsC9QiwNtXIMfd5x7qSYgmnTWZPjiCyC3cW3QG7JWHGLY3EAQbBCkgYr2y5kWBOl/wFvYMJZcTlFO5fSbLrfeQv+02bkyhGaBueTiLg6DMCmBdz9k74F3AaTrojpQNDBnTWRRZlX5mOINyd31YB3F3nD7kL4kJ9hGTvmSmVEtSmOwCyQQ8bJX+dPVmVF0vovsHfVjIoxw2bri7IpdUBoXrFr5CAYT8FijW5gXceq5kSYGYUWKQyM75B+1JgeeNCbnA9Bj9Hm1B2VHGX9qzFAfyT/yQAuuyuW0j8NtEcXziw4wJ3jdpKZUbRZzRj677pKdgGHMPMNu7tLANoKQX3Q69ghooclTX+Hp0fo4SqC+NlldGYzFCzZosQURTIQpT4qtQogYp0QT3JluV4QB4PPeQn5A/6jZ2xn9GxA06DulpYmI2UNKKYPUuyjA2oRinrgfLlDmDzFgw4FS/fsQNg7YP/quOmG0AdMf0MSO+Xplp/yAGs/w9KdbPbkbqSqakhgxVOBHLsuY0iA3AS82LpdmGvlu1Oa3xCzneU6SsNBS58VR9ZoQNS6rgmEJHf1aoSNBACEB4+uPBZ4jtdCOXgtddjObC+UeqJ+BwQOguxHg5nIIGWe4E+/NZjH6CLAqLO8aEIcordZUobUK8MulsRBBlzJSQAUhqfw0nUZfQoNJxrnz1gGcr3Rs4bEqo17hkJniCnQLmt5sUB9rBm9eu2AFytGB8pNIBtqxOvv81bL8AECWl3g1cwUOQQ90MhS9NnqtPK8W5DYfxQnaSqCYoH3V727D5hiR30r0v3U7CMGV/ofiUR5YcnmLIUcFH0EBo77ijOE/9reje8IcmRjlmKzrcMEzajEW714NdG5mwLlMPFs4GmRCMqU3RZfJqTHuJABMg8K4bJrUgGqDgseGh0C1QQNuj6wl8I+PmG9X9x4PPWAqZQ+kl414wsrZodxHMtoS19sCeeibwN7gxmIzrz1L017uv9OBhSwKePjy/YBZRPgqGJvnvSL4Xr4FKwG6KeU2Mg6BL2Nk8aq5p7KDM6cyxc/TwY3qvbygFhheAoQYfZGgnMZeTGpNOmjK+4fQVV2VGhdduFf8dyTc94SFHS8vFY/x2rk4JdHCB6ssxlWcsqaVCr2dtphgPlMODoXYbkj56wO1gnZJ2XgLxqlvpI6x04IDqQ1xCBhyNBVS0ZchMZnDmtHOFSij+sIPy44qja7RkZTOxjYTYM4D7wyfWpMh6JzchDvTDcViyxc1C+QDfcU1gFwMVdw386KZ0gNq0Qr372M5BMJnsholk4yIunjvFcoV0qHWHWQKiAVRjDGSyo6EZ7SCNEZTFFDwmPsKNBjMNNjiqR/eoStjYsbXq5oiIv50rWPIoiB4V07eYI7Fxn00bEMb17BX9bDy+kAotumHjJsY/70F2xg4BMMVBBFcNoz2FylChcu26sHGm3BUYW9bjuYdGfBa0pLr9vTJe7TZTdmkC0sU+Rm+nGjUk1VWYrsqzFCdoWmDnlSRKP6abTOhS9ETA/DV47sBqQrjFGOCOhLwdLFeS9SqI6ljrOAulwTCAEOqsGsXbPplA4byo5DizVnupWqn3DSK6MjLgvWhceuZQGKaQNsygfJR1UhyTwYo7hqqdK1a2bmw8imegOAg+bVyq7CJNjYsyedHxspHFRRX3JsaJrucoBNBQTzjDXzE8stKAtS94VzIoWUIG2M2GT6e2eQ4qw4jnkE2oBjmNaGISXJCKVqmLU6tTnSkMwTP/v25SvAsfwUzbaDwp6jHgUpACRuYLZ5kroCkab3o3q3z9+ocIzhJ4KCLXPNIFj4DZDTRdTftBJCWLKA+mi6J70ryf1A5yYCqIt6FwuCaDiAM8UjOuS5vR3TH6RuEYGBmclN7YaQfRls2ApCA/x3I5GI0FtpMBCFO48gpNA1BTjWh/BSSgnYRbchAyzolwpu4RlJiZqmkHMrK9RkGDyyidilRE7rs+BDmUhk7WcND/TezeiM7G3EDCMdzVx4apaTHcBCFFhSFyPstxAFknm8DsSJaijIstd1MhqAR9K6J+QwB3UBNSHvkZfzHnKphzM6Ru6PBcHcxmzJiMldv+gRUceAx8Mzks+0ZAcRPjBEoC68R2hk77yQkGyNCtPkIbI9IPaqkTeEA38tiHIIdwEvIaZRhgm+g7OK4WuhtM65GAJDiNMiSrmZ+tZzkONMaRBRQb4w3bw0x8VwcFSrc2B4WQ3h43BUcM6g05+snBcoYGjFJoryBdoK0yGlTatbNTx0MtJe0NWBPg9ea8zv3sZDmoTgZZKNp5jROea3AchDor3KuCbUKCQcNyOOCsp/c3vxojZWVDh1lqQC+/fqbvbL0v1gAAIKoKeXe/el/95Cx5X8x+biim00W2DjReJ18b8YWXTJKNBuXN6yl+T/u5oqZOFcYczEOdKRoejHOAjDaCvhCDChP5KB2ICHL6yaomvWnn7urtgttBgDCtslZ81KbC7yt9NKRo9cS4n4NiNivGAI7p1EfCl8DpBBldiXYcaNnNrieCENn6/izFgbiTRi9rrxSIh14Uy5UZ7FBI5vcCJI9SeHpkTJ1b9N/LwabogFvepiR2jFGPAv6W3iTS9YuiMQORVRTQrsUI/aWKpnKQDeVDHVhLDyygzFmsGsjE3i10gYoOtOcLgrbHgw91RTujuMSdeO5gI2HExC6oMvSrNa0DYHkErPOj1Gcp+pGRvCaFZHA/KjXJukPgQgtAlRLrKRZUIWwIoUBhYpwjyjTcMVtaZLGZho3yNhAWGGLk34NMajC8TSmR+e19k+I91e3eYbkvbDvzKewCFACqTOh1gxOmZmZXqW0v7KSzoPRjNUyIJNikJcUr0Qac8FCaL28TWpCgnYc8toHlYdb/JsU//+3htTZyayU6rQUVDb6BAAFug3xhycg2mFE2kJnbo4PgSvaGWf9N799BtrQlBwuebII5BeEdQS3IaBhLQXne5DiArmxYGuLsBKGAj/HMDYdV9pa9j4XKaRuD1qQ597PzOBRR8iC6PHB2BWBiMxUN3UhrDxxyMRoD9g9AWuNg034QaSmFU2Ck6EhXzn9xgbmVPVswRJWgNt6MgAXtAYttTIyDm3w2zDM0YmHjjHgS2j0jZDFus+h21FXa0rktMAxhKfvJhKNRhimwUJZ05Rt6Xs1GvNIuvjtGTfRC4ASvzad1ez/Id7cnU/KQlfnvJd9gVwLbCbozStGF2AwFIpaRZuZnMU7w3cFFCif8GF6p0mnMaUxNpCC5xeidtNm6EXol2RuSfRysastLFepSiQGJFSxCelEAkxV65hTTLcZ/sZvPSUa28VItEya1SmbZ12McdDOfO5hrGvviomFxSqFz3RVHff3+hwi39buTTUJPOJNiVQPkkLPY3hifBoVCNxmWgZU/41mIA+2eRK1WYQr73GpwH+oIICtlI6QzbsBxkGhELV3d5A2OfpIZc7AyqduQROy4Wgy9KWjo0gfJuCLxLJC5MtDLmEO+yXHAUIAhGzYtTHznsEvKnMyNzBZkC2zYYAGXAn2AofMhxqm6BJVkS+8ow8RoYiMf3LDh50g2Eux37BCZ8CXtAEf3k8yYpDabpG60ixlzAtemxg+1awD2ladXcmairrgoBzGYw5ZC9GFuKva50Le2zak95iGLsdMAnYCRPCA+/aXNPolisV518kFoL2OmOowWHZaLmAwGClqZvKI5Wh/PPZR44sDZc2VAq+DBBBdrpPhtR4ApYepkQxsTgq20ZykOUB8aSKFbMS4aoRbZ0AqfNY/YaAK7BT0y4+EejsHs+xyUZkFfQG20hU2wHi24HZrW/UGQRKFCLgNocCyb6i+xYFYUe1ImpgjoXXEj1t5g+/Dhnm3g4Fn5rBeTgmu7HytYyiADXGJHTdIprGAisWq/XtB2zigbqAJG0+CFL44y6wdZMOWt4A5jS0ts77ClKRSEjHalxH6CTbdWTgvsmUPMxsHdPmWzYgcE/li+3te2REoHsJUdDuefU1xAa8Zi1ZsY72mEAqmUFaaEv2qoAivG2JrJ+G0K2vVpsESsAzXcx4MPmQXddRgzFLou54uDO7mTZFDG3xelNaU5W6MwmOAo61mMI9vYK+g5+YhybUVkTr0Z7bkuSAxKSxNY6QmxsW6li3FwUFyJDVfdqvQBnSGGL3SMJWBA7JiK1iHwRhRKjnmT4j0FqWkj4jNnJlXcReggUiXflV2IJ1eWtBu5YmaKZD8efGgUOFPtGo2B9B2hJKdBJRIWwR7cowrj2OuLF2O6qD+LccBJNEJG6AD06seF05fLXGTGnTqE9/0qK9oGIzVjxKzlOLjZBzdsG6c6GPFwE4rxC5tiGPorEU4CdoOXBUqi7I3QcXJSXAaYab4JDHvGIjAWgrO3F5a7oOpltyvALvrUKyro4xzrJXAZqrOd5THN1aMMdtXCqdxSazEQzNpWFmVO2AXccYxyMtcorVqNjvqtBxEyEQqjpTHdhjxiUwbdyAGZgGyXhTKjHrUVNRkztIxo7A01qjfYNg1d4qhglsQtdt7qMlfHE43XNvsYznQQkYxg0iasw1LSyvAh/gQBkmfdfbGp4ev3PzTHobxzMaBOGWgExzdLqIZzjPSAGU7Wn5FvL/a4PAtxoKeDzg+oxSkLX2ueJuySrIrtAf5UgAFfLYAefdLCf7VjASULzqkKQt0X9NEL9DFQOkZfZ8Cp2O0ErQJqGaWQcRJ12W2Tk1QPwoq2LhJUqCYMzzJiqqNCMse6KxqU+SHGIYIqyit12fqWncOLyFBMxkqwqDUGQcGT0xcm3HWY4TgIurTprd3JdtEE9yKwP+HhjOsk8PJ12NR+scU6bjRPTokzPgI6HkquSDsH3KQykL0aQ5gXUalyGtoOvvG7GCc2qg7rKq6UY1eesp1qPTiKqrE+cUB1zkg3xCfe8RrnZsaHIvzNeA/s0eFBhnG0sjtzDx/YNWAsa7/Zmd6oFz2LcWBGWEYaXii2SjkCw7nUjKwZgtSLcr5WU1Ncb/Mh4XEShNmwmQQTba1Y5MLgHUy9MiQ4TzcbsEeRCLF2dkW8OY7yHRJ916rUS0cSFQpHCbP8HAqOmAyEfAWoptWVxkOQQ5QCkMQSYVHbjy5YNUoFXWOqZAG04iNpi4wo6HEHjY+DqMy9wcYmQ0+tSE0bXFGKNjYzUTOGyDfzzLBG0ap0/uBxEJYZtPoTZNeFWFaIzjIAktMR6+KK/iutOQjzZXzrTYx3pCOlGKiPAG6lAJrZeifWgemN7CjT5Ebpe7CbVd4kPx58pkzRJ+QRrUuE0gJGwwgrPpX3EyUrqt212vrl1aOuP+YLPVHKQBvELXCgmHcq8OqXYqC6HfNOkO5QJcm2B9keexCFCQCTvdudWnLQ10HPxZorps9qFE0VZZAIyLcyuR538iQKkwbx8q1GDtVhDnOwehVz0FPsCqy285atFZDLPaQ4NMfB/TJOYGYQe1gGm0cE4ASszycBk83725anFts8x0kEZmFdpo6jWLztRSvbjmB8G/ThoojMNkecRzIeCJNjH2x0TKalpzKuEiUaEJhWyWWgp696AeyGYbVpr5crMz0KwWRsGZ4qJmRztEQH6yVpzF1FTjJTCjZwz7MCaLaHIO+E2E3CfT0dx9iCnOrrZ1R3YSOCXk2RBLMsDLg8QHbjJOiyM6hPbryU4wQb0QB/WmGGSlHShXTcjgi1hd4NOWY6iLIDuITVkJ/0qWlSkM0UDyrCMorYMMV0x2BQki5lucnxnv4G4WyB6kVG0nF1EBdD0ETBNLYe6Ixyw3yz/XT6RMs8WMj01TEwg10YbYhXQQLD8j19oINimgPKKXtu3/A608n+RpE+Ljt5R2mBvJxQfTBoN0ZAkqkrcj/hY6RoY3KcLGRmo7Fo9h8H1FiPdlCfkreuLRZ9YaaAwsrB7nGT4j1VbZnpad5pRmmG3KvydoAmRC++2nwg6zSZ6u0+KzwPoiz5jlI0tucE4efkorJorUDcFcAVSzsAAkKN5EX9mc8WtekyGzNTi6Y4WOPCkL/S0mvRV4cBl9li2+huYhwsY7JggnvIfOy1aLeDG2OgaMKvGFvP2IXNokYg046tmiermLSjl02y0NkqQZ/NflsvsOberkWzsLJ2ttwGemaeQ13a8mswI6PEhldDicPurzSDJRXuO20syHb2MT7ru1XnwSpmJVShiMhMy8WFCvfHBtjUr/JEhe1OURbLX8t0KerJUY5tS6YqXAY9muKZxfUL6uzlK8sJNXAgtrEwjZiKnEcZLyfM0IWldEqrWuz9YmoWVqYZukH+Je9RIC0eaPNDjvfyVMkJFCio6LDlfv/MtOGb8q2xkyoIlQHVj7biiPb85ANxhLItFh1Dku6UYdQkWAAOvs6m4bwrCwZ4KeEh2vRsZx7c2gO0LTOvqysaswvbF05Dq97gEvQIlxo82874k+E5Xlnaw+bPufh6TN/68ApLtirukymWgIBWiLWhbMNMzctMHZwYp9yhrwYLVejDZi8JsZ3toQhyImruOJnC2pzxLMSBPIMxN6kbeA3PsPRSFE4B84SS12uGRtwryWzpVPLS5TyIu6STQb+HySZHi2UWABryU9+eVbdRrtPdgP1vMlpzk+KA39hsBULp8Qoz1nEMVixE29TLMYp74XWGvMu2fD4EOeQ4aCB4/7k43I+hDRYiZuqEUC74GKDVuaEyr9LoCLBPrvIZtCQBtVFgdz5c2UVS720Nhtj6BpBrWEWv9+Rrdec4uNSJFbpeJeq+XckWRM5iA39ShcAGA2Fnvy8swp6AztfoLiEyKQAIR/cmvW9ftIEaaKtizxv4NliasKdlPJ57qMUBks5mDY0fOZYCci2Y8IraNmUh5cFwJC1z8jcpDnS/OrB0FsaC1L8CTKknk9p0GAKBmhnzmc33PvgU/5wHd3BApwLLLSb5seBLQR35HlzIKzwXW0sAT8uEOL3DfKVMuYw9jIJxAu7pLiPZskWwXKO7f4aJWekQlPZMxLV4Defmx9mVy2gj9eAeHGZzuUngCs4e+9ipxdQZzT9Hdc2Dq3poclMz3Sw5bi0qERARE29zSWLfRYKZZ9iMTyw8m+tgIcIINDKvItXohTJfU41NAf6TGc1QufVto+aUaNZNjhNdHiPWY4hkWR+ykAlx9NDtU9F37cgE123YGpLlfch5EngpBe252rhnoPWdCgemsBEzA4koQ3Zq2gIl7wzPdZJowkizV7MKmbc0WG0GQJkdZN7mSLALVurfLGRp0a+fBwuXBg/IRBQ2a+u2YqYGR+v0VgMMG1Qk0FGQyu0mw+uhtq0shqyMfoHzChTwlwN0Ud+xYhnQSAd7BnWM8pD6kOOducYiw2l4Apmh/vyRLEWXtc7fktx0Z+NtuimPkuU8WLJUqrttDyaQBaftYk8Je4k4khVTqFTNNnUUm2aQiUeOdQ57ad/TnEGnubY8DQLaJgtJRyu7PkDUSU7M4C6xRrvJcQCf3n21OUQFPpULZzytJthgYJoI2AQ9avoQg519/SHHoUmODYnF5vv2HGV+5nrgHZGC+iAHUBenDma0Pd1kOMC5oXC/siwVNgFv/WXW+DJSUSlkxmwwO2MXI6PNmL7MUKyDTJgWrbRsXaUWvc8KURecOVAFha0gJXDUPEDx6a2WlU+SCFCcoicKJr4E9q57ZVvaEASthBeAGZh52Y4vWvn9toIVG33a7r/qpO5PnymgeWvfKuhdzIKBjAQ5KlvRnx98AHq4OlVJowa/poPterZmJCNlxB1l/G5mm2WRGvsBHCxiom24LH133caoS+TB7usG9+XaV8htU2adGxL7YdbJDeS0myyi7jUIe7mbEEg32Qx9GLkoIGna5wvaaa8WrXNQTHlnqumwx01nSaLaTMQLjB/owohcNMNtwAIuxWQ1rWc5TuxmocBdDW66vVCRttvRyo7VoHZjkhhqa5iNVgvqk1VPwviBmwELZe1cdL94nt6R4gg5/PDp0JxRuyArjNtSX+MzW+hfH9TTo265ZAw6kkD/H0VtEjLmZrFSPre9zmExcdsMp1cCylliuxSkSZBHUX4o1yVtAzgNCwHnXs9iHGh4SADIAcDH99hNk5u1ubiqslqxODUBma22FyNW06yDDJiMbTRkaTnGlilqA/JbVtCqJXDsHfJa7pKRrpWbHAecKfxpEL9kahSuGwvKIuoEAJYvGE1nYQf90xFwpnVumDwZl75l3+Xiuuu0QHRdKLtfXHfdyN6Kbwfsvo9wnaxpJlv1JkdCi35dG0NZAgIp/y6xuKgU9EVxzqTwbFKcg2MC0AbJhMkGmXstx6ZlyzR/WtEYbdA9QG5Gm7J5+Wz1l0BVZKKFv+j8eYGfQUSiicEkWruqNXDVys9YnzIuaT83/NWtqq44ouX+YFxnvmmA1l6xFnIY1kt3o0M+XuazFO9CVW1AOTDrSNnNlyrjA6ABounRkGZ+mpiPrb4lVkGucZDqsMDo2EHfNm/Go5lYcP5BzBA02pSQDS0ACUe5y3E/gOxPcDnq229/94uf/wwszowhZdNiTC7hNNlIuVgJSkwXU5ALGuVmfafsYzbr12uY//hKEl0xmUUoR3qsaKFGxrj6pLjqTNbeTaee3YJhYv2igpkf//x1ykf2JE9jevHQGSOwqVGxicGJX6hUQDOJxQIr4jX0NQ9Gk7nCoA8X8IrddrbXucGkl2CKDb6RzJB2gXhnc01ucrweTlK2tl2tjZFPDydh1lvZKiQsFA4G7WpBRmNcOgbf1rlRcb1z2gd6iOUbQSnBLuHNlpYRHpMxSGssdFtw1BzHtQ7WMDd8oJTA4LAO/QAgItcoNQCkH0Q4DD3tBm8nLsXEOAi+5FcPW+MNT4HrhyJGRk1w5VG/0Vll3oy15DKjujc5DpxH2bbn0IZXmleqxsB4VYLYGBdmYp5Z8gwNvuNw1zn+SyM2oQFKdnvNiFI1bRjS9SDRhhOHJR4gAKWoz1IcCKaIY1kNYhtLItNgaZBR1cqOXWxhNknqhD3TWWrXQfLLzsA8ZHYUjNxsQvJCAUn/gOYvqIP1X4YxPUttZgQyJzeOL2hQSezI7opnHrZIG0ZQmCdiiKHzr5K0UFbz4YF1jvySpTGjQshktfZQmM5WSKicSAh8KHCzGoCAf4DT9qnAdXCfD9eVxGtII1ts3dalYC8AzO47NsjCheMkZtTCuw/a7IPklwmSsMKmbeUc1wqXae3KaVAWTwjJDtOGgq6zlcC7QftgUXOBHNiwiMHOlK9tT8DnCziGNgOXKW+H5wXSLBM2HnIcGvnRrxx2FUkqgguTGyp3RtBzLapwI9ZbMeRFf5biQC+ISUD0QOe+HacsN8s2G1IBCGO9N2ZrGdliOyFwMuOxD1Y1bQ0cEGB90e1LM5dxZ+HEIGTYcRxA35hbrdZQrzc5DtyWbcgJll3VFjDNZHxa5Cbt2n0FIHDQrWHx0XxIcQrUX3go9DSrxiYwpRw4EmYIRuwjY2emEVnJ5bEGyBtCO59cz7DYxEb3bQd3ESU26LihP/PTyEhEBm14aicG3eXkvAcLp5WAwVQUXXQSo2I8NOxWDr9SR5AoDaLydpPjxLyHwWzkakdQtJApgrmYtrfecVhlwswy4QLojBnMhxyHqhaZzXc4CwVBO/a8sFwYWuHG8HG4lcZeaL0SHF352XEcQ2JZb4xUuaWA96MoEKOONILkCwNTbTNZM1JAk+JgeVNxJoMdg81HObdgKWFACZ1lxtidfdFNIrEB4L+9F7JfK2/ar9fV2LYnMFhKaMixCAlSFg9AizWwBzoJx+/jwYfqm4p2YD+tuCov2pChLtthalunPBpvEOZ0m4Wa+SbEiU3DG3YzlA5r6LPmjNkrCmUfc95BlOrsBBnryYi1yXEOotnpdTHUYPFEieFea9xx9ONigW9GVQPfEqVWz6F3e4m9aGajI20Ux2KX6oTDa8HD0dIFqEi2Bp7xE0Z7H889U79iKxyr1kDdzHxxF1ElYU+CIp111a8ms6HVpg4c87PbSfaiXkiMMUo7OJQbUCdGM21Q0A+jAoQarMgiaQylPIfRNDgcDTraXtOILxiutjvKsEfznVNMMUO51hcsCTJXNzFeL2TMOW1x0LZav6/4MZoavTBmPWJOsJC0gFVlLELRUH0IMs4MxiXI5rwxXWI1ATASkOT0ArxYkagwwhVKR5+rEwHnQXJMnMEYufuaejcW5KZYBWrutcX0qu2AZmkfODqfuN8nYZqsZ4TNl16Ec33BbgvYheoG5POO5ybekbFg8y277W5yvGvbLFga0LisiQ8uChYtAW2B5iomR5vhj4jCrtU2+yRQsyajVldUc22bpcYJpZSCKAX/LXhKDMgA7diM7vU+CdRkQwmsZrSmgyo02VAYrCiVkQ+vfjP+D3Bt2BZWE+Ng2ROFZPcBXF5rBODGiHsURA2ww7GImf3pQJ8U9w9fG7jnWbwNAA45MLoiYSygPWaEr2ySZzee8iA2yDUh0fl6HIfqWkBdCsFN69fVAF/DiojMDXH4bFLkbbBR3wa7bmIcKOQk9n3bHgJpxQ5TkRSG2/QqLBQBuRmEvnI4hnpwl3aQHxNGhwJPLutDYmm9UrFldOsDHgKXo9A8VkghF7dSDO/tl5aUT6NapPMxHL5OP4QVYEB64BLuMRg2GQ4ZA1xp//r9z5gKmiyJPIzXEIxGULCzB45Vm15PI8yBAovEbM0gJNwHkZoNmnmKe5MYJ3gRFdDuwX6hyfoOjzLJxphcA6Tm5cW9zykDa6ca7mrAPh8Zh6XFy8Y7PTenlMYqpMZUEhnQTYx3uA000N4w9JCRf+JLiwXU8JsFQ6jRl0ovrYjx9cGnEBWWgrJ1TFlHAPyLLAEmmsEwjzLBUpJt6QxA1/ZnKQ7AdinU0IWUv/RJDwhsWchH5BfmugC6Wqy0hIpc1/Jf/hdGaNNS+NAAAA==', 'sha256': 'dae0fb50bcf18acb11eb344b516adc1be111395b87786a8a7c20adf9d5cb9cae', 'bytes': 53496, 'original': 'coupled_policy_audit_by_state.csv', 'original_sha256': 'dae0fb50bcf18acb11eb344b516adc1be111395b87786a8a7c20adf9d5cb9cae', 'kind': 'unchanged_saved_output'}, 'reference/coupled_policy_summary.csv': {'data': 'H4sIAAAAAAACA42Q0W7DIAxF3/kWGpE4JPA1CBF3RQo4Ik607utHo2Wa1E7bA0LC52BfJ+QSgwy0ZZYTZkoxe6Yii2eUu583FAvNMdzdyvVplb1Sx2mb426UmOL1igUzu0LEzgeOlCUcVK2rUcsfTKCUKLtME36j/Qkb/QQvMzI63GOdLqBbPN+ehPURgue7W2iNHHd0BQO7mmVb3S0UdwSRYLsvzRop8pawan6u3gcWeu2YM0ZXDXzzv/+uTlKKhD6/hs7tPbjWjsZqq6DtoB8GM4hKx7SlP9RL2wDAoHWvDICygwW8tCCSf/+HXRuDHa2yrRlVX7trKz4BYxSL0AYCAAA=', 'sha256': '5895b944b183bc57a9667337e17067e2c2d2ecb74b1bdd21e3a81075cb392104', 'bytes': 518, 'original': 'coupled_policy_audit_summary.csv', 'original_sha256': '5895b944b183bc57a9667337e17067e2c2d2ecb74b1bdd21e3a81075cb392104', 'kind': 'unchanged_saved_output'}, 'data/rgb_natural_historical.csv': {'data': '...', 'sha256': 'fc753183ddd196491c8b4e790b3b2ff274b3ab9131bfaa1cca301baede68616d', 'bytes': 996645, 'original': 'results_rgb_natural_calibration_audited(2).csv', 'original_sha256': '1f6a773da9c818109cc5778e150948902dcda741840ff42a5b7e5625a45b39be', 'kind': 'column_projection_no_rescoring'}, 'data/rgb_balanced_historical.csv': {'data': '...', 'sha256': '14cc7bfc0a62b3d07e99710d447bc73856277f6afca9648383edb7b93ab550f2', 'bytes': 987506, 'original': 'results_rgb_balanced_calibration_audited(2).csv', 'original_sha256': 'ab551ff1b86d843e1a0303d3c1a96edab94c3a3fa7051ac059a35ed3f5c67f59', 'kind': 'column_projection_no_rescoring'}, 'data/cost_sensitivity_rows.csv': {'data': '...', 'sha256': '8aa6c5c5985cf4cb8406ec0a9ead5ec83dd9177df5ff4b6c707acdf6e776a024', 'bytes': 56312623, 'original': 'issue5_cell_results(2).csv', 'original_sha256': 'c06a0d755a3be4f1fc04208a9bf5c8a1dee0af5dad35acd7dff07bf61e09fb29', 'kind': 'column_projection_no_rescoring'}, 'data/issue8_calibration_repetitions.csv': {'data': '...', 'sha256': '5669f75b87f247cb7035c8d6fdb47f5bdda3047b26544289409ad82efc301f7c', 'bytes': 8416637, 'original': 'issue8_review_bundle(1).zip::issue8_calibration_repetitions.csv', 'original_sha256': '25274b6b0c20287b3482add29a0c0919b2da25fd324ce8a25e4da2f671e84142', 'kind': 'unchanged_saved_output'}, 'data/issue8_known_law_summary.csv': {'data': '...', 'sha256': 'e7dd63d859842ccff3806c64df2300bf5c9e520c679f9f6d267d99eb28679a6a', 'bytes': 952260, 'original': 'issue8_review_bundle(1).zip::issue8_known_law_summary.csv', 'original_sha256': '25274b6b0c20287b3482add29a0c0919b2da25fd324ce8a25e4da2f671e84142', 'kind': 'unchanged_saved_output'}, 'source_archive/ROSE_RAG_Experiment1_vector_Jupyter(2).ipynb': {'data': 'H4sIAAAAAAACA+09a3PbRpLf8yvmmA8hY5AR5VfMMrfOcZxdV+3aKslx1ZbEwoHAUMQaBBg8JDE6Vd1/uH94v+S65wEMgJkBKMpe7128WYoE5tHT09Ov6em5/YYMfBpF2WBGzr8h5Bb+z5+4+W5L4elg46WfguQ6HjjsXRjgQ+/x8+eeR5+Ihxuae4GXe/Dq9o4/ypIi9aloFn5/S97cbGkabmick+mM+Emcp0kU0YAsw9hLd4RehQGNfUo8/7cizMI8TOIL0S0ZVN9eJ5utl4ZZEpNkRd4lG6geOeQs9/LQd8hfXp8SLw7IKfVzL74sIi8lqyTdwBdsMSNQT+l8FcZhTsebJKARoTfeZhvRbELeF/m2yDMSxn5UBJRceVFBM4f/HSercQmtX6RX+MaLs2ua/vBbQdPdD94yy70wJim9xD4d4tM0D1eh7+WULJMiDjIG5DbJckBKko6DkMNH/DX1PwEE7xIAB17C4AAKAIYghomXUmgVEJTSYAJIQZws4IOhve/0PXu+fPYCZmyP6YP5i6/CNIlxAnXTAvhO0oDka5zAPLyi5GSXr2FAOM642MAwfS8aR+Eyxdm+ommGqOkzBh8mR4BKb6hfIJ5cH7CYw9u4iCJ1bD8+pkero+lz89gSPrc4uIV+tOFmm6Q5yXZZNT7xDEay3REvI/G29W4LQ4U38N82aL3M/HC7az3dePk2SnLAig6n2zSM8+HFgGNydjFwEKaJwN0k20ZhPhydHy1G7Trvis3JjlWJtxPXFXVcV1P0hMHNym6DjrJnfiiaZQPqKP23cnisSjXaWr37EfGT4+mzYHq82ouI/8bWOZJklkQIwUOT3/T506Mjf7o6hPxWabIhrrsq8iKlrksErXhxnOSciemIRZQCbpbmSRIpZVh7CIUfeVlGM9lg+ahRdFXEPmtCFozSwvU94EyNgoChML6UpX4O/dwhb6F/bxlRh/w1zOD3GTArZJQO+VAAb7VArl1arB9Oask2Dzfh77SECthimlzqWqy+vUpTb0fmuAiAyPGHvfy/l0gZQte/03j+IS2oQtrsHTljU3a2pf6seoUfFwP8n5BKUrTxCSYZFOdCAFcvL1mv7UXbtTcjqyjxckK+JSfDN/Mp+U/y9/l0VC+5BMIqC9ZKHjVK+iBjRMn6i9jb0BnJ8tSOkYCuYOF+oq6/Bgqk0VAF0lEBGZHxnwhDuBYp9WenFKgbpENL/ItuyHWYr0maXGcwptsjZ3rHlq2fRMUGpPgb/mzS3VG4IrBuyPBockRezjmK8csUfmOL4jmOQzweNcDHf6kXZpR8RPH/Jk2TFJgbbwmbYHU3RYY0CeQZk/Mjh0wXMMWNuUj5oIEYGSkO2/2ctx+xxwjtmHXEUb5wrAUZbA7/oyuqexYgw5vzaa2/HXVTiBvQANY303AkpWRD+WXGycJhNAe/JFM4B+pbMLJhzOFclELOwV4tGjORURrDYs4oiL36m090m7tVd6wF1toCyp8vNIUFKGVfjXL1GqBBSsLkg8BZ/j3cliMUQ9ORzieKDCjHAQ5h5lNU/2S9SepdwZIaAbkcj0btukC8WB0JGDrE4c/0846vJl4QDKH4SF+khqKJt93SCg5bFTYwWR5/jExYqsgb1F//07DWI4yxaq6boJZFGAUoZgFtgZv5FLhEmGSNJcM5q0pRFWdu0ni6TiTXqr+g20yyUlj/dPxCeW8gTvbBHws+yP4sFguHdBGxjkn9hKMFZsjNEbAX+LiJt1mGl0WY75DmJya8fwClG8wHNK3Q4gD8kmwdrkBliCKJImhyTVMoEu0mUCHMCGhC4RKEdU6jHfFTCl+yersBxTkXRlmagNwTkoyZS/gL9H3AGir90GKepLsx2FYZfEVYkDWuQD/aNanEYJnFFEaMpO4nYHeBODAOmAuPBrRj/q9BIIJwKt7QpvSztbelZBhXVAar2RV4c8gx/Dea6OjOBf3jHzAaVAi3AHOua/2EpmOBNlEcRjkUDBpZ+agUebwNxmzQfB2fvj970+g45ppFv+FYRyARw/lgu5l3+BxNbFjVYMvSoCRKQWP9ZC+sOvKSHPUTqViYydElhZHCAC89NCXrgrRJTZzg5zrRiULfIezjYiBQB21phN9wDH075BH7vBhsaZaBspnloW8oz0uORXmum1qKj2vFo+R6DLZ/iIMb4wq0diKBWoeX665qRgmWetcVfXcJyL1lI7CD3HMFUfMfXFfh0+NKkSlmS0MNEjYNNLITsYpQEAq2r5daXDGbc5aMItePQFLzOhP+8lEdYpAADuGaE3wdGaQhU/MMzbJ3j2pD79eoHLcUsG1VW3CJkW4FKXNaSmgpfeUbXc2aTFfmyLjMWnxUKDyAD73qV4JRA3Ik9SRjPybG2iV3NXRT0Ywbghi7YQQIkocOIxqLiYN50VBRrU9ddzYqlZhq9NnC4CRDRo3eGwMdS7Ewb2P/vN6LUxvnQt8cGxQ0NuQELJo6n6KpMhIjK58yA8ZEs8Kk8uIdTjMoGX6UZHTIOgC6v0FGGF+OGDrkL0QER6tpuBXiJWGyXxoYTERSq2emsaYM5e4BTq0ai6y2IlUuIrE+4VJcy6IWPSwpRXXWLDPTYJ3WOMTa6tatQVPbusyVDc1SP8S5c3fT0qZH9zUocNzPVXoPFFVYlEPR0JjMNMw+MWGMDUr2x783Wm1gxvO5EohyfMCd6e4ROg3LX1Pxi/vWLwaNBvhymxP0PDID+xKwM2TwjPRmOKd4VuKcLxxYBgKO8+ZKMuFSTkKwbdDOFv36ClYbXpmGZdz22Rhe6edGb7E0J8thPrpztkoFI+OP9CbKmxtAhhwgCXZAXaGPSuxl6m0Yudf8W1nLpdVYdlIhhUlCBswG2ZiZDd2AkYYgQaHbO1NTiHgg3CIFyk02oCWWDoaUboA4gN/MBA7YYCeTycJsmg+5bc7JgS0LvpjZV1zLrIuR0rjZUofSbAx6FicID0ucQ2mjSoX/8iT3olLdYCBMsmIz1DFlXGD1oufTBfmBt6FjoMAA0HXLmYBT/ebUjxKnzSIMUkW3tttFR7ahLmmW885EzxVshsI1OBXI28Vxry+MC2Zi8iazWWsZWInNqEqUpKbHjByQEKxI7ec1MW2QrjG9yd2y8dJ1ZBaaHCIEsQ0b0iV/+m/z2gj0zVmnSSKidNVCR0NUFGwSnY2GkSSMhP/9nlRqjAKRQ2ZO2fbC3KIsgiJx6S3DCN0jpVpedscXi2YRWBC50jf+JzQhLWPkDB39VXLOJXOqwHEa84rbdvYWeVuP5nqYvld77BqcZhHUSRFA7tUQYIiD9VJdtGP0mk2PLRiqrXDDwtYv8NXFgO2pu7cqwHdmMSPFCGOwyN2rvh21bUeHFZ2R1ZNnp0mSKzOvsbmk11/46KWC4JTfQBorTnidR4CzAm5QlPJ0NDL765URIIAOG8c+Wo2LO3AuqFx5c0w1wfultBs+HpOMb6pBrLZ1h06lAUHabMvRi0kRx0m68aLw9ypIZUmjkK46FJ17KDHwib5Zt67MyKcWpcY0yj20m1rfI02vX17bqYHUT+upVbFrPypDcPW6jlarGe2rIbTn72vQFMyvaypEC3ir3Px6dIxhh5xW6eRQVWT0r6aldOkUZullxnlTs7HX7RxiTVVAE15drpx/9RX9HNlV/UOku5ChUq8DWZ5xaS5wUBPcWkvRJGvXfrqv96C9q/XPdSM4pO0k7eViqD/7i9hDfF3tIaphnE0PRHJF03LPtNpAqO+U6vpq+SW0+36ukA56v+29RX4UfgIlYp0kgbvx8jS8GYqt01kN3Qy78LFQBL0utgb/VU1yV2YeGqXAQyihbGuqhrWprthe4tKpcXyJEX3fynC/1/nGZw7pw8yNzKBqv4/q1m6mz3w6ut57qLUMsWZdT3T9hV1VNfoz0reW+3r+uiEhq/pMNno3YdaKfpMO4ytuazcbYZLQBrDYwMDFEO+GvCWTgvEteQN8ZgfMFGPDcxH8QLwsCy/jjOQYSiFGSH6naUIUSCb6JmvmqdidLt3aDkaKqjvXXIo1dVHT3sw+81aGn3NFusL8OUcJ7gGBatJCsHhtbVu68F30r6u7phvvZqh2rBuI9PhrKwvG1dFEuRFRA8SpN921KXHwxkSX67Ucm3Gvolan9IpUuxSazY7F/z9f61ftM1UcdUhGt91Gw6ynIi75ziMyHOql6cgxSXA96iyF7R5LvdGmL3/XhbFSptObLSqAcgUCqSOLzerah6quPLxtiUdXPMByt2mp0xt05b6/v87SmhO9jNzbLAXEBiG+6hhiiY0OomqJjI7ySZHP5eS6iMdh2VMX/V5jWONcCKxeprq5kJ7uHnXOfLeHPjvfz53Q149fChBzqxp/krnwIyKkrBYVUvCP7ssJe+xDtMv02JTQIOaQHQobDJ97u6LLK+JYCoidjd7INxl3e+lMHbbfgfsoXJCPHNPWx+F7HbX4Ne7gz4ZSe5HHmbhrY9GOgUH7bKaFgQOcJWlOgyEeWeCR/6ojnoX9173xYh9q1MNxpARS7+1AMobZVYH08os+zP1f29P0C4al1yLRQQKhN2kLKLkJN+KIliXenu2IidjojEXAV9HdnIjQEUWWeHA3zHwvDTI1nn5J82tKGzo0tiLqYuw8/lQD7nmw/SWNWdB+QMK9/Vyd8TdCH3EBdENI9bkxQJXRsfKgioITxGUN6xVl1EpN4uwZ6XyP7TeOdBz1rL269w0qqhprBsdK5lKVGBk9OVWZLm/yZ3PrCHC5eijd3SpcesW74WRBUhMtmfTsr8UfYvVZyDHcz11hqf2Hp+IPT8VX5amolngNKS3G2I2ge1ncpWdPBuCXs3G+6GPEscBULkWhO7HwOvZFlfOdbO9bEYN1A2XWvTMKI2XBJb18BqXKrbKJUbUvjYH4PSzHugknRl81Mt2rkV6mssLl1QH32YRuTXR/NFViyTA68oMKTr9GR3uCW54z4AyxfD7qhbjWGqs8AyZ9oer5n+hrtMDdPwLgC/oQrwFdueq9AHSMQQKF8aoPy9qHgzwU96gD27GHdRCP7XAH/mszLeNgFfQ+6j3IB/Lt7c1cdfSL+mR3F62ajjr0A8NzHsoFCTPegrNTa/nDl/iHL/EAX+KwjHb5nP7EjGWicxkSU5ptwfqhX1t0FysUYAIsD+grAC07DZcss1W7wVGf/BUwWFC8QEsBzYudgwd1CbSYHfEu0UGWE4+ovfDYrVV4Q4N6Tgmrz43PI1bn05JBq1tMryBDv5j9fQXsJEkn5BVPTMGnjoVp8DeNY4Pcece5EEDDA5EjTHaRrLhfjw+kgMUHUtbz1xJueXL7j2CzP4LN/gg2+/L+yS8adqbxIXlxEAaYNUfvKWeUW3fEKRFWM5YhzE+2O9zcepfE1Ej9dfddvY2jvm0YdQ7V1Wou1ZAoLFS8gUQJiLkRhNDwWldt8X/U99a2qlnc4LyHIW4O+DnCSJ9edrgVEpihAwGZ3gcQAzMAce5yoW03SnX2hYZCbWEP1UScTxfdBQFPxnLdNlTJOaQnydxhh71ntBG6AnQq1HaUHCqYOQaBUo3/eDFy9j4bY0XONQ0v16CFVQ4sA18td1iCJB/qFVmnQvO5OZUJc93Icsx5U87NngwpoxHTIF19IorG0EYj0/6NU5sbju6UpT2sQDuvd7bQhfai0urmKcX1PCyPvIrWRgeakI3RNOFxOtezY8pRU4L9rx5VwhIMowEIFMlQU5Ip2/0SWoowt/rYWGfYILqA3E2BB6PcLXnJWtoCTMWfGCmjvSJOuEm7RWRM3dNKYZVDyo0UqXsjAivQm6ZJvYlvyUeog6EzYKNuCnfqgHLpwJcz52pC/gZigeXyHV81AEuWbNOfRfbrwj2BAzddwmWV8/EUaXY8NXtOw5j+VngoDMQo1G6GpnFjn6OOhlAbmymVGCQKunpU5+DXoG+UTdeNMFgJr3FB24Y77RxlE8ij9hhb4LaBrYzAhRFOkRUfOD7f6mfaLErxOoTnQ6ZEitfm03pZEeVMq2c5ojVcrCQaDZ955RbLeWuGNAWXjYIwYm1z9Ld5n8bUYvqmOJbm/I+jY935OgnmPFdg1tICjNgXsRocb2AA+cAFM2POxtMCLIGNzNqol6sXg9OS9QmXGLnELIArL4xoMCO3orMN9ORd0ruLwR5WvV7kMwbPGoXlpJJpvTKC0XBdl9VgCXbl6KqqOwY4evsIg6QABukmqedHX6WPME8iCsIRdwkNCXIZ1YFMwrzzPEh4RnieuadHR3t6EbmE4+44TD0lLueo3HugACUblo+BX/fBMqSyCTH6DV8RjuQxR7JYIcS7SkJgN1SEmaM9h8lytS5M1Eaa8hPdkRshxNLvMuYyJ9Lbi07FZCmypi53zYOrUO/yvk5Ds4+wtbNqXCPAkYF71qQqnkBXfhpTfYap4tSW2RM63d0qOTvm1J9VIjtdwA4QsuZ5g4p1JfTWQU/GKBRSRXNlMqqOCLAvFtpqjHI05Y+b5ZNlRtMr1KfLPm4bGcNbXfKc4XfNBR/DcC/ZhqW00buSt5ZLt0pWWV/R2oT4lWbDSeqKJ1DUIMxqbBRgAKcuE2iOmWQNUqankj26Tz6ye1F5B6X3pXYbxfej+t6U34mMKLmWE9Sx09thjtdwdz5djHoZeJfetuxWoRUyVgGz+2nbS8Ludrm1OUvLVXExmFUrx7FVUcDGSirF26op48Nqyk9rNUAYFoc/1mLCdNi5wj5kXdBYu36dzhCcXpQESibO5su5olZYdwEsvq9+SOy9AhQStvAxxxa0WDF8x+aobNGicw8/GQfI1d4s0VxmzasltPOiNIj5LhvSyHgy/p281wv5YxQxLQovtfLXoPoQnkypfpDij1nuP8vNaWD3e1QzNTJ6/dThSFbXg/uqo9NXO7YYR30sw4sB19sbOjm/i4v769m5lkzRSSLQsfNJ01TocyVNmiwLgJ6l92JZ7KqtswZs1YtDzjDp8rDdN8PFfbNb9M1sIdaepaVGcoihbf9173wRVtqvpWfW+DjC2KyAtLJLONaimlwS91a5RntTKPqTXeW6S1e5JjL76hJC9TDg8W7OIucxN+OIXuGdJjRd4V0mYYx3vYgljaqMeg8m28Iag+Efj3G11htlQkCW5mfM5K028nTJXta09bbGh4/WYZPcM2jmIQJmOoJljKsO05DxDQ4cmjYpWc260eWq5zaLPRt0Y5lYThlJiM4fL8wKDLshrsvC1d8CUe33zCwBMh1hAroQgUbre0QK2A0ZtvNaR1jWCAG7V0Q426yvkahFJ2vQs7ngA6W5sIYrNJBRO0phGcMeqoE2XUpVukfq80M2xLnRV+Kx2hZfmDwp9rL7XVmiOggNpCPYu6uwd9dyjEGtyuMWkOnzedj79MNnzRPaIPPGhSuL/TO8dMft2+50qa0Ue1NfIF/pZ89T2kVW/Y5v6LrvrvV9r+zXpn9KXtR+Fbru9ln0bMfmL+yI5utZq0+SqJ7HcTpe21jD5536DrFy4OEZLTvvjgHTH5/pWCN7SVOt7LgvYJbZu7cIrPuf0XTT4lIXVqV6jdUGtGPWNcA2I13URvucdyq11uNFV9Il67hRsT3AT13TMwezujCxVfzrT/t4m1+7f3l9ihUaSLJW+vWnffzgVXp2No761TW2iqxpLBcyB3pX9jbF3W4t+HLeoohH7EjYkb0eGrqNip0dqbTb2cnIio2UymwqipdhP7yQPzUTMcjDcPcFKk7ctZdu9gRJxcrLB4Non+0M4aLaBpOfvdz7JfU2dIgLtofTp3I2BSG/+dYt4jB3c5rlQ+ZHZCkXWPYjnvmg123qIOeCwufREx4oL7uc+eWRQ49XGAUKIKchvcKbvES/SI2Y03Yv78m2cgkeTV5YfUisyBONB6oWFXZkiwqTR6k9dvi1mXhoDhDwXEPwbWpIZbQ0VZ2WVV90XnvZdUeg0cnIa5yX41hYHJKNsstFx6EIO4Xex5WzvwunGakY5iEozrbUMW2HTsexYilMj5p7FsKI5zd3JCu3tAW1GqIBtnHnAV1zaKU8M6jNs3Ax0PeI/A2JT1cDeYNbLlRFTrA6T7R1Tj8mb6QQHh9NHjcKGY8HgiAq2JUre8Kuf7P3cNpY7xydab77jhj9BZ/oziknzi2z7sknkzCnm2yocyJUOythxu875RhkYefNNmEB5Uk0ZwLb5JHgO2avsgxlXxLboilFROXPknGjwCAoMEQ0JRvaLQByNyMXA2sbJdHe1iG+c6qQtVtlZHcTU4NdwpE3MsDnC/i4wxK3+Gvg0yhy0YsNkzrYeOmnILkWLQzCAB8eB0+eedPnz8XDDfDpAAQtvLq944+4PgsPznnH335L3sCA0nCDse9AXavwshBbiEzzwlFKG6QLJj8JqOia3lC/EHywiHN4GxdRVIP1x8cvjpb+UzOsSZFvizxDYBd66NmdU667KgBz1HXlHgUIgkSkXtUhW5T6R6YaVqyprZevo3Ap2zmBn40SMFR2iI0XQJXD0gNYl9soyaHFyXaH39i2SZS3CsbFZrvDl/G29a5rw6X69tOrszfuz29PgTkh5BP/Ohgq1Pb+1w8nv34QBcqyP6COC8vSTb1LV2BcJd2q1mTzKQjTIbuZ2E0+zT+khjtlvgW9apNcge7koc5GGEnRjER0lWNcZxLhkXAMPmB518R5cVqSoaJY4QKF4u6W34SsAHMZJcvhxeD7Cby6aO2yiTqTIo7C+NOwQ79sBBl7V1SEAbgc9CH/g7k16QZtqbRXWC60Qzzyt5IIBCJIEkcsXkQc0eNuCPMR/vd8Uho9jsnJz7/MMFhkRdNU8LL3gNSIejwP5rZYsoRILf/BmJx9/POMh1vEIa57vg+Tbbmlwxd+EOatI4i6YW6DlYvrBohKobAfkGneIr7uJlCiWSm7uuyoBCXM2jRH5ARnCr7qdDQBlKP13AO65yBmESzdYYFlcgMGv7+mGZTK8cSW7URAL3jkeG3wsBEfDo/929n7X09fvzlrHxE8Y6z1DEhAA35pNhzp4BOGxJEh/hjeHf2oeYe3bsOAzqYunrpwxQmO9tCcewD6/KkZ0OOnFkAfWwA9djewKorNw4L6zALqYxuoUwuoj90oueYmiA1I7RbXyenb96fu37mp+lxxnLz66ezDm3cf3r5/575+f/aBvX+svD/9y3v3z6dvf1bt1HrfaLXyfKTH7PMp+/wRP6fs+ZQ9n7LnU/b8WI3P7xMcUsRimzKmWeZGXgDCpqWZtu1pcSc9M8DrV9KLJdM0o9YhTb3UX+9czV460+euEur22mdP1wl2JPGnUXyHhwVFm1IDm4xqsZHT1arsHalOV2gEI18WuDMASuAWFO5qY3QosCrz0gIK7NtjEqbKQDdeIS//SUI+bJS2DaLGitg3Rrxn7EiPsTxEdPyBg1HTqleDsiZb7zu4/aj3M47RtAz5KQeba919oOeiK3N47Kg6dWE7JvdV0VXtyBy3/vedGbmY6z4wlWm0HWFyCdbrlBtR2vIqPTfPQtbpX1//0LitGqr4scIfddmNpGxi2zjmsI062l7Oa6RMHlVdGaI746Be4+WcqNs7PapXxV/ONdi1tmFnrjX5fMCWIIgm5kZbJx0nQ1pCTh4OqQtK+9bOR1fMCVauTU9HNT4PfK+vkx3JSoB9rNFv//Gji/PDcNGcJ2s9Hm2qGxcj9+miT20F0j1q6QDuU/0kedXEJJjN/afjJPn5dQ2xULv3tJwkp1okQxv9Jqqx/Bkg9UcdO8dt8aEcayypuiaKHuq0UzvpXKWJy2PHVGqPJh/xvbKt7hGBYAkJKbX+7jCQW/t2bU/Oo8YtVBELE2Q1nbXUbYIe0UF77wHts6vNIdK02HfX40HCH+72dNqX6woIrLatXRc/o7Zp2KxQEk53X5McFLLsSjNjNWcaBi9gfoZMNYirRqAFrb+JEf38Fy/KqN3rhRDvDwoOcwlY8YKwyA4DQueEQ7RG+aRy3Gbh73Q+/NEhT5ubp1gOXbNDi/JwLlahTlyohRSxrSuKuzU0nV8MEu1oI29JI3j7TrZhR/zDAi4VhwPgVtJKfFnYmVZwAODA+/4JUHOt5ACw8Xr33nDfsFrDi8GrzRJWBcYS88V3MdCU3snS77d5uMGwF3EalaUu0FbJwxztywECND599WdSsZtq4NqaEb1EEalt83Kdu5G3S4q8+d6yQXMx4N+0HE8LAt+T5rUejrUoWgwTxOQyBeOAHRFApsl+LXfDSm7rNBkL0bEj/tiIjejUQpWkN5XsIL2K/NRxdR7nwzF4N+sojCkmqHo48oSxkFM2yeQj00aSFXkjA/2tVGqoxDYki4x4EowvTbEoGK8y1zL4XrQqwghKqncYyfVznJdW0yX8eWinuS4g7jINxdG+CPcfPZ9RyRQd/y9eOORZMzoNaN1Qh+8VOOSJOfuwGB2jIEOSzFreYe1rS9JfmVP0yPp2an17bOq3SmVsvM9PGSNqZUlAZ0pQJOa11oct8Tqz5pk8R7yoW141JI76B01hpXYuPY2LFU1LOc0jhyfXKGllpLMg+MlOzMTdO8iv36aMtEDF/kyFhBK+h9+qeQhH8UPuuyAqOP4FMtiPOjqU+TEFbfXb5WAd8tuh2M1QTo87pzvw2eeMTz8PdecxlfIGEuFhspWSmSEVKlOw3HEOs77Iz/nP+2ZVfiBfhYyKRWubfXce0Lsh7iOE0tbrVaoKCVdeXa6zzohwyD+g4d+OYd/bFK5J2y9nCoNcfTppaayArktZI9xk6+R6aJIU5lycSRpehvFcJGHSjsZjUUZQxCtyvaZJbzCEen5uSxOGvGaCmyYmFqEW826MxSQjt7ZVFTK0tLAbYX4SJenSSwV25c8hQ7mh7CSjuQs29adMZFlGOViXvaOuqrycVWLXFPC3cU7TGEw+zuJPhn/H/CpWLbyXzi41b2lsi8uM/ue//lsEOVP4KhQrIlaDoaHDVOz6grufit1WmmvZRz5DoAlqH3MWIWNa7sP7JYd07/msl3qBioXxeIqCMSEu++Zz6dizPiQnpl0L6DWY88qgX+gNDMU7r5/9nrd1q73ewx0MCK+18E91wyZ82stlUvI7FcRmreswYOGjGM9nlNEAATJaXc4u2eeYt6Sjk/rE4gFPnQLBqs9NjUiP3V+r3D19fHYdUHdD+9rs7OkH8BuWShi9LOrmywPAjjcY9sP4rwdh/Ncq/1EfqKWTSmMBMYOkucyV7aXFJIwSH+M9rC7yZEtEhR4+Wy76qwk3shngtOKsw3z61CaqTzvdt3jXg8KG62zay5GlszjfL+wYU8Dgh1cPd46pI+sW8jRDYwJ699OQHQ9lPrqQ7djNaoZAU/rvk2Xp/n5iXmfO/06yJBWHvUB2ciNrZFqOLFUedxFrHMU85chCF2wV96r/ct5owJCmwGztlldFs85KZzfXxEfGu0jxxJmsNKGbrSmJBSgIFGVO7MW9surynGod0CrYUQBG+8IOsFLvIWD+TKkRlKQIHSEGYAV6We7KMTETqvRui1QJ1uo8dbeClmYTbDqsTdRWbJG5IZo4VzwIqfuKqOEtA/LOIbesq7vFRcftw+ppR5ggnm1j1J3goFaJ9dVRiVEBJgLIyTL1/E80p4ENuIc5vS+uB7Gd368K7q2WNidLDCx7gEgFcRER66ib4V8DINTNis0GM6IbgjDqnN9ph0ZYC6gSyFqwzHbAgZ+10hxYck1KhMKU2WSY+Rg/x4CLipFxw6Tc9X0NemKaRHiu9qcQrLAd7rHhxVs6vZHXBb2IfE+eHGtfGuq8Vs+ozrSFYOmOyYnixJiRW2k03plrvCoP7xOey/O2YRJa6vKjPjCVt2i4yPC0O8MYTENbU9D69GMycCvs+1Sz3U544C/P9UZT6Z8xn3PGk3fLJImqwKXz7xpBg98tJl4UDfHyBN2lM/sB/SqK+O1iykqQ2Uo57IrG2QPsmn78nZLD5kGhPpFZSKpsINWh8lRmExEJa6KdHe7G0u4Nn5l+xNoe88kmbKdUclE9WUl2gMwak4DHl0OFq+7V+1t2XtmLc8Kk7BaYrmWNGpGMV8QIv1LpzuG3XONhV5baOwOwkywbC+dJQFGlwT38iRHfSswMoXHkpZc047dbZ5QdSgYaSa5ZJvZ0GeYpsi8/2SyBkXGr1dxwspJenOqESoZhq4RH+uAFhEAQPoUBsYilycET/QEBD/Eeb79ius1rf65K7jshb3MsjtrCjuKl4joILgb8xPB4CahcYziIlPiooWwwaIKpG4jwDOhmCX16GQFVNeTJ7PWNZsWWrRh0ErPry9kBcsQ0Co1sh8d9s4ltX7vh7GxoD0JITfKbHCydCZfdOb3JtYnULy5AFk3+kYA6XpNuOvQDGpIAAAUpVeSr8Y/7nrpFVQJT/7Wcwo3YDHSB6k8tNpSKdoSG3a/ack5bVQtU7KxJlkwzch3m69a0SKWu3dCEJV6AqUpg2Won6RovbG5jv+mFBuJbhZHuMgzsYRIUm+2wMUaH1XCY5hjn82PzmCwqYH0KNXYWzGmHd0sXxdCAtO9NSylemIr+ujKLxxQ5F/BPdoUcv5NuVUTRbtL2m7DKIItEjgE2bNCBSJ6AJlPN6J2hpvahEuvYN+OTMJMd01slutZSRgayWoqweFHLex6ZaSygnkUxFmqdvuhOR7UwiN/+SMdTI2bVZGaYv3KdZO2FUi6R3rkFwPh1mVPAdcl8Tr5zXeR+rvvdrHnvHfLE+2W48Z8+f+75x9O9MtycFnEjxYgO/A/s1lUm7E5+/uWHs49/LjOXeCktlwYoliDE/qOZM+U/Jg+dIOfF9Il/vKT+IQlyDkH18unzH4/8p8/2QzUXJTnD5UNj5IjS53R5dHxwyqC3J7t8jaQfZtvI28l0PuKnjjh4wyDN0lp2nc70OdW3ii0EK+5DSakXMA+J0ni/UxvKkmRho71b1By+UNpCCbUHdNpNRC23kDxKZ6nWeZOYgqGKLWuToNCdNszITN+kGF1nax+TN8wgaIQ26BsV6J+sAVvD46PRPdead/z08fFj7/Feaw0TDAWSRz30Wlutlv7z42f0c601YK+ObcGh5wJ1NbZTHsZNzcEa6D/RpdIxBFrbyzZCtKxl2/tUrfLNnKRC/xLDbKnnnMQAU2WROSgJ9QVZVu5BensRAD1+snr25Mnj8erxlI6fPH3qjZdPXgTjpecvnzxD4eAH9yMOBuI3BJ/XqsLjwSeaYtjGFsTeTAxB4IFpFggYpyci10qEZ2S9S/Zqy16JF7I8f/h4wHtF+AbxkmddgtdP1N8uKJlJCk+ffnP3zf8CG3tTLObbAAA=', 'sha256': '1b4041126d9d3cd56f81efbce8ed942d2866fa841fbdaea7acf6a4faf60c095a', 'bytes': 56294, 'original': 'ROSE_RAG_Experiment1_vector_Jupyter(2).ipynb', 'original_sha256': '088b2c90d0f4d210cc89267dc424bdcf83171cc61d7ba24035fc89f2b7c874c6', 'kind': 'source_corrected_v0_1_2_original_hash_retained'}, 'source_archive/ROSE_RAG_Experiment1B_ANACONDA_SAFE_v2_Jupyter(2).ipynb': {'data': '...', 'sha256': 'a9d42df61534512e1e4c61b9bbd1d9d9945d1a9c6b2272c1d0ec53251f2071ce', 'bytes': 94739, 'original': 'ROSE_RAG_Experiment1B_ANACONDA_SAFE_v2_Jupyter(2).ipynb', 'original_sha256': 'aae6d6199f24b143315975819c8f2a68d5c2207eef360b3d2971984a1d494f6c', 'kind': 'source_corrected_v0_1_2_original_hash_retained'}, 'source_archive/ROSE_RAG_Experiment2_FiniteSample_v4_Jupyter(2).ipynb': {'data': '...', 'sha256': '7eefa8e150a7f3dac5baf6c2f31909fc1a23d8b191a8883dbc492fcf98ad95fd', 'bytes': 105421, 'original': 'ROSE_RAG_Experiment2_FiniteSample_v4_Jupyter(2).ipynb', 'original_sha256': '4e7ba259cea37b218cc4dea3b253ea1676b5562514dd5e861cbdfbf143a52dde', 'kind': 'source_corrected_v0_1_2_original_hash_retained'}, 'source_archive/ROSE_RAG_Experiment3_Misspecification_v2_Jupyter(2).ipynb': {'data': '...', 'sha256': 'ba236284af83ef51b1bbd6bf53b5268520cb1d80f13746f30c73eb8b12efdefa', 'bytes': 129867, 'original': 'ROSE_RAG_Experiment3_Misspecification_v2_Jupyter(2).ipynb', 'original_sha256': 'ed57813f155e0e7c5b9dcc028f7a5b278ffdc5248bb2837ff326bf29b6ee259b', 'kind': 'source_corrected_v0_1_2_original_hash_retained'}, 'source_archive/ROSE_RAG_Experiment3B_CoupledRegime_HCR_Confirmatory_fixed(2).ipynb': {'data': '...', 'sha256': 'c8979118b2fcd908ba5cba545cd1f66d155cfd21fbfcf4cf6bc5b7165a4f84e2', 'bytes': 93765, 'original': 'ROSE_RAG_Experiment3B_CoupledRegime_HCR_Confirmatory_fixed(2).ipynb', 'original_sha256': '3d12cddc9ca4374129307619b36bfe4c966c39ead953e42511cf293277d7b2f0', 'kind': 'source_corrected_v0_1_2_original_hash_retained'}, 'source_archive/ROSE_RAG_Issue4_Scalability_Benchmark_JupyterHub_fixed.ipynb': {'data': '...', 'sha256': '6cfefb78432402426a5ad717e03721bffdc09dbdae3c54a3df234cd3434e2528', 'bytes': 106233, 'original': 'ROSE_RAG_Issue4_Scalability_Benchmark_JupyterHub_fixed.ipynb', 'original_sha256': 'dce0c9e0e87f94b91812f9589fe397082606c2d74062cf4085af4f62010df885', 'kind': 'source_schema_corrected_v0_1_3_original_hash_retained'}, 'source_archive/ROSE_RAG_Issue8_Joint_Dependence_JupyterHub_v1.ipynb': {'data': '...', 'sha256': '3326a1b5cb4f50f9bc7e89d27398ec8d54406cddf7c0655d1240b5d24278d85b', 'bytes': 123490, 'original': 'ROSE_RAG_Issue8_Joint_Dependence_JupyterHub_v1.ipynb', 'original_sha256': 'a00ef4966f382927718f73650ce970e951e9c9da4ea2616ffcfeb2961118fc43', 'kind': 'source_documentation_and_comments_edited_ast_unchanged'}}


In [ ]:
provenance=[]
for rel, item in ASSETS.items():
    data=gzip.decompress(base64.b64decode(item["data"],validate=True))
    require("embedded SHA256: "+rel, hashlib.sha256(data).hexdigest()==item["sha256"])
    require("embedded length: "+rel,len(data)==item["bytes"])
    target=RUN_DIR / rel
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_bytes(data)
    provenance.append({"path":rel, **{k:v for k,v in item.items() if k!="data"}})
pd.DataFrame(provenance).to_csv(RUN_DIR / "input_provenance.csv",index=False)
del ASSETS
environment={"python":sys.version,"platform":platform.platform(),
             "packages":{p:importlib.metadata.version(p) for p in ["numpy","pandas","matplotlib"]},
             "execution_scope":"saved_output_reaggregation_no_solvers_no_API",
             "generated_at_utc":datetime.now(timezone.utc).isoformat()}
(RUN_DIR / "environment.json").write_text(json.dumps(environment,indent=2),encoding="utf-8")
print("Embedded input files verified and extracted:",len(provenance))


## 1. Finite-sample certificate: explicit numerator and denominator

Coverage is recomputed by requiring all six recorded true channel parameters to
lie inside their corresponding recorded intervals. The structural model family
and the prior are fixed as in the original experiment. This test alone cannot
detect an incorrect joint-channel or prior assumption. Certificate exceedance
uses **expected true policy risk**, not individual answer error. Both all-run and
coverage-conditioned counts are retained; fallback runs are shown separately.


In [ ]:
finite=read("data/finite_sample_rows.csv")
require("finite rows and unique keys",len(finite)==6000 and not finite.duplicated(["n_cal","replication"]).any())
coverage=np.ones(len(finite),dtype=bool)
for i in range(1,4):
    for param in ["alpha","beta"]:
        value=finite[f"source_{i}_true_{param}"]
        lo=finite[f"source_{i}_{param}_lower"];hi=finite[f"source_{i}_{param}_upper"]
        require(f"valid intervals source {i} {param}",(np.isfinite(value)&np.isfinite(lo)&np.isfinite(hi)&(lo<=hi)&(lo>=0)&(hi<=1)).all())
        coverage &= ((value>=lo)&(value<=hi)).to_numpy()
require("coverage flags independently recounted",np.array_equal(coverage,bools(finite["true_model_covered"]).to_numpy()))
require("finite risk columns",np.isfinite(finite[["true_hcr_risk","robust_bound"]].to_numpy()).all())
metadata=json.loads((RUN_DIR/"config/finite_sample_metadata.json").read_text())
tol=float(metadata.get("risk_tolerance",1e-9))
gap=finite["true_hcr_risk"]-finite["robust_bound"]
violation=(gap>tol).to_numpy();fallback=bools(finite["fallback_used"]).to_numpy()
require("violation flags recounted",np.array_equal(violation,bools(finite["risk_violation"]).to_numpy()))
require("published aggregate counts",int(coverage.sum())==5835 and int(violation.sum())==27 and int((coverage&violation).sum())==0)
finite=finite.assign(recomputed_covered=coverage,recomputed_violation=violation)
strata={"all":np.ones(len(finite),dtype=bool),"covered":coverage,
        "uncovered_nonempty":~coverage&~fallback,"uncovered_fallback":~coverage&fallback}
rows=[]
for label, mask in strata.items():
    rows.append({"stratum":label,"denominator":int(mask.sum()),"covered":int((mask&coverage).sum()),
                 "violations":int((mask&violation).sum()),"nonviolations":int((mask&~violation).sum()),
                 "violation_rate":float(violation[mask].mean()) if mask.any() else np.nan})
table(pd.DataFrame(rows),"finite_sample_counts")
by_n=finite.groupby("n_cal",as_index=False).agg(denominator=("replication","size"),covered=("recomputed_covered","sum"),violations=("recomputed_violation","sum"))
table(by_n,"finite_sample_by_calibration_size")
table(pd.DataFrame([{"tolerance":t,"violations":int((gap>t).sum()),"covered_violations":int(((gap>t)&coverage).sum())} for t in [1e-10,1e-9]]),"finite_sample_tolerance_audit")
fig,ax=plt.subplots(figsize=(6,3.6));ax.plot(by_n.n_cal,by_n.covered/by_n.denominator,marker="o")
ax.set(xscale="log",xlabel="Calibration size",ylabel="Observed model coverage",ylim=(0,1.01))
fig.tight_layout();fig.savefig(TABLES/"finite_sample_coverage.png",dpi=180);plt.show();plt.close(fig)


## 2. Historical RGB: reaggregate each protocol separately

These are saved, benchmark-derived channel results, not natural heterogeneous
retrieval backends. Their scores are not harmonized with the later cost sweep.
OracleReference, when present, is retained and explicitly marked nondeployable.


In [ ]:
rgb_tables=[];split_rows=[]
for design in ["natural","balanced"]:
    df=read(f"data/rgb_{design}_historical.csv")
    require("RGB unique task/method "+design,not df.duplicated(["example_key","method"]).any())
    sizes=df.groupby("method").size()
    require("RGB 495 tasks/method "+design,(sizes==495).all() and df.example_key.nunique()==495)
    for col in ["wrong","abstained","answered","correct"]:df[col]=bools(df[col])
    require("RGB answer/abstain partition "+design,(df.answered.astype(int)+df.abstained.astype(int)==1).all())
    summary=df.groupby("method",as_index=False).agg(n_tasks=("example_key","size"),wrong_answers=("wrong","sum"),abstentions=("abstained","sum"),wrong_answer_rate=("wrong","mean"),abstention_rate=("abstained","mean"),average_queries=("query_count","mean"),average_acquisition_cost=("acquisition_cost","mean"),average_total_risk=("total_risk","mean"))
    summary["protocol"]="historical_"+design
    summary["deployable"]=summary.method.ne("OracleReference")
    rgb_tables.append(summary)
    task=df[["example_key","base_dataset","base_id","capability","split"]].drop_duplicates()
    task["protocol"]="historical_"+design;split_rows.append(task)
    table(df[["scoring_version","calibration_design"]].drop_duplicates(),"rgb_"+design+"_protocol_labels")
table(pd.concat(rgb_tables,ignore_index=True),"rgb_historical_summary")
table(pd.concat(split_rows,ignore_index=True),"rgb_test_task_manifest")


## 3. Joint cost–abstention sensitivity: audit all 400 cells

The stored policies were recomputed in the original sweep; this notebook only
reaggregates their saved outcomes. It checks risk arithmetic and the 400 published
summary rows. No new API responses are generated. The coefficients 0.008 and
0.012 are experimental cost assumptions, not measured monetary charges.


In [ ]:
cost=read("data/cost_sensitivity_rows.csv")
keys=["calibration_design","abstention_cost","cost_scale","method"]
require("cost 198000 unique rows",len(cost)==198000 and not cost.duplicated(keys+["example_key"]).any())
for col in ["wrong","abstained","answered","correct"]:cost[col]=bools(cost[col])
require("cost outcome partition",(cost.answered.astype(int)+cost.abstained.astype(int)==1).all() and not (cost.wrong&cost.abstained).any())
require("cost grids",set(cost.abstention_cost)=={.1,.2,.3,.4,.5} and set(cost.cost_scale)=={.25,.5,1.,2.,4.})
require("cost design count",cost.calibration_design.nunique()==2)
require("scaled acquisition arithmetic",np.allclose(cost.scaled_acquisition_cost,cost.cost_scale*cost.raw_acquisition_cost,atol=1e-12,rtol=0))
recomputed=cost.wrong.astype(int)+cost.abstention_cost*cost.abstained.astype(int)+cost.scaled_acquisition_cost
require("risk identity all 198000 rows",np.allclose(cost.total_risk,recomputed,atol=1e-12,rtol=0),float(np.max(np.abs(cost.total_risk-recomputed))))
cs=cost.groupby(keys,as_index=False).agg(n_questions=("example_key","size"),wrong_answer_rate=("wrong","mean"),abstention_rate=("abstained","mean"),average_queries=("query_count","mean"),average_total_risk=("total_risk","mean"))
require("400 cells with 495 tasks",len(cs)==400 and (cs.n_questions==495).all())
reference=read("reference/issue5_summary.csv")
require("stored cost summary unique",not reference.duplicated(keys).any())
merged=cs.merge(reference,on=keys,suffixes=("_recomputed","_saved"),how="outer",indicator=True,validate="one_to_one")
require("all saved summary keys",len(merged)==400 and merged._merge.eq("both").all())
for metric in ["wrong_answer_rate","abstention_rate","average_queries","average_total_risk"]:
    require("saved cost summary: "+metric,np.allclose(merged[metric+"_recomputed"],merged[metric+"_saved"],atol=1e-10,rtol=0))
table(cs,"cost_sensitivity_summary")
table(cs.loc[np.isclose(cs.abstention_cost,.3)&np.isclose(cs.cost_scale,1.)],"cost_sensitivity_central_cell")
grid_keys=keys[:-1]
cs["gap_to_cell_best"]=cs.average_total_risk-cs.groupby(grid_keys).average_total_risk.transform("min")
table(cs.assign(tied_best=cs.gap_to_cell_best.le(1e-10)),"cost_sensitivity_ranking_with_ties")
del cost


## 4. Other selected evidence

Scalability statuses, failed/budget-limited runs and solver/evaluation measurements
are preserved. Prior-sensitivity and coupled-policy outputs are reaggregated or
copied as labelled saved results, not recomputed policies. A reference prior is not
an independently known true prior. Dependence violations are recounted against
recorded risks and bounds; joint-law coverage flags are not independently rederived.


In [ ]:
scale=read("data/scalability_rows.csv")
table(scale,"scalability_all_statuses")
table(scale.groupby(["method","status"],dropna=False).size().reset_index(name="n_rows"),"scalability_status_counts")
miss=read("data/misspecification_rows.csv")
table(miss.groupby(["shift_type","shift_level","method"],as_index=False).agg(n_rows=("instance_id","size"),average_total_risk=("total_risk","mean"),wrong_probability=("wrong_answer_probability","mean"),abstention_probability=("abstention_probability","mean"),average_queries=("expected_query_count","mean")),"misspecification_summary")
prior=read("data/prior_certificate_rows.csv")
require("prior unique task/condition",not prior.duplicated(["calibration_design","condition_id","example_key"]).any())
table(prior.groupby(["calibration_design","condition_id"],as_index=False).agg(n_tasks=("example_key","size"),mean_prior_shift=("prior_abs_shift","mean"),mean_reference_prior_gap=("reference_prior_uniform_gap","mean"),bound_failures=("reference_prior_uniform_bound_failure","sum")),"prior_certificate_summary")
table(read("reference/prior_sensitivity_summary.csv"),"prior_sensitivity_saved_summary")
table(read("reference/prior_calibration_quality.csv"),"prior_calibration_saved_quality")
table(read("reference/coupled_policy_summary.csv"),"coupled_policy_saved_summary")
joint=read("data/issue8_calibration_repetitions.csv")
joint["covered"]=bools(joint.complete_joint_law_covered)
joint["exceeded_recomputed"]=(joint.true_risk-joint.reported_policy_bound)>1e-9
require("joint flags recounted",np.array_equal(joint.exceeded_recomputed,bools(joint.reported_bound_exceeded)))
table(joint.groupby(["representation","covered"],as_index=False).agg(denominator=("repetition","size"),violations=("exceeded_recomputed","sum")),"dependence_calibration_counts")


## 5. Export a review bundle

`validation_checks.csv` records numerical/integrity checks only. Passing it does
not prove a theorem, certify all structural assumptions, establish scorer validity,
or validate an unexecuted solver archive. `input_provenance.csv` distinguishes
original files, column projections and output-cleared source notebooks.

The resulting ZIP contains aggregate tables, figures, checks and provenance.
Embedded/extracted input data and unexecuted notebooks remain in the run directory
and are deliberately excluded from SEND_ME.zip to avoid sending duplicate data.
The archive contains selected numerical verification outputs. Scope and provenance are recorded alongside the results.


In [ ]:
save_checks()
require("all checks",all(x["passed"] for x in CHECKS))
save_checks()
scope={"version":"issue12_selected_reproduction_v1","no_api_calls":True,"no_solver_execution":True,
       "coverage_recomputed_from_saved_intervals":True,"true_policy_risks_recomputed":False,
       "RGB_full_annotation_checkpoint_included":False,"Issue5_final_v3_solver_source_verified":False,
       "source_archive_executed":False,"issues10_11_reproduced":False,
       "source_notebooks_require_separate_user_validation":True}
(RUN_DIR/"reproduction_scope.json").write_text(json.dumps(scope,indent=2),encoding="utf-8")
bundle=RUN_DIR/"SEND_ME.zip"
with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(TABLES.iterdir()):z.write(p,p.relative_to(RUN_DIR))
    for name in ["validation_checks.csv","input_provenance.csv","environment.json","reproduction_scope.json"]:
        z.write(RUN_DIR/name,name)
print("DONE — selected saved results reaggregated; no experimental code executed.")
print("Send this file:",bundle.resolve())
from IPython.display import FileLink, display
display(FileLink(str(bundle)))
